> **English archival edition.** Saved outputs are historical observations, preserved rather than rerun. Chinese text has been translated. Use a new run label or output directory: translated source hashes differ from historical checkpoints.

# Conflicting Context → Agent Action & Memory · v3 FULL

 in  behavioral_v2_full  as amended.  **Llama-3.1-8B base、 Original weight  commit、bf16、TransformerLens 2.17.0、 Real inventory / Reservation / Order Tool Process **。

 Only two pairing conditions run in this edition, each time  prompt  There is only one authoritative file and one search record: 

|  Condition  |  Authority  verified_profile |  Same clients.  retrieved_note |  Rating value  |
|---|---|---|---|
| aligned | Berlin | Berlin | Berlin |
| conflict | Berlin | Paris | Berlin |

 City follows.  source  random;same  source  The two conditions are just...  note.value  differ. This is neither four concatenated groups nor a setting with only false records that requires guessing a hidden answer. 

Actor  and  Writer  Both explain the facts according to source permission; scorer only from  verified_profile  obtain ground truth, then process authorized updates in record order. Tools execute actions without correcting customer / City. 

** Its a complete, operational experiment.  notebook， No new model results attached. **  Local checks cover syntax, scoring, genuine tool environments with mocked generation, caching, and experiment branches; 8B GPU  The behavior still needs to be in place.  Colab  Run the authentication. 

 Run Order: Select Support  bf16   CUDA GPU（ Priority  A100； Actual need for visible storage also depends on the search and generation stages; mount  Drive； Restart after the first installation, rerun the path cell, and continue from version validation onward. Hugging Face  Thats what the account needs.  gated  model access rights. Tool simulations do not use external order services. 

 Default  pilot/train/dev/test=24/120/40/120。 confirmation  test  Close; developments are completed in  14  Section will be separate.  FREEZE_AND_RUN_TEST  Set  True。 After disconnection, rerun the same configuration in order to reuse completed case outputs. Changes to code, dependencies, or configuration require a new  RUN_NAME。


## 00 ·  Independence  Drive  Path 


In [ ]:
from pathlib import Path
import json
from google.colab import drive
drive.mount('/content/drive')
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/AAAI_Workshop/agent_entrainment')
print(' Results parent directory: ', EXPERIMENT_ROOT)
print('v3  Do not read old  baseline、head  Sort or freeze experience. ')


## 01 ·  install once; restarting after completion 
 Keeps the old book dependent on grouping. Remount after restart  Drive， And continue from the next version. 


In [ ]:
#  First use: Run this block and restart after installation  Colab  Run when. 
import sys
import subprocess

python_version = sys.version_info[:2]
print(f"Python: {sys.version.split()[0]}\n Interpreter : {sys.executable}", flush=True)
if not ((3, 10) <= python_version <= (3, 13)):
    raise RuntimeError(
        f" current  Python {sys.version.split()[0]}  Not applicable to Ben  notebook  A fixed dependency combination. "
        " Please switch to  Python 3.10–3.13   Colab  Try again when running. "
    )

numpy_version = "2.1.3" if python_version == (3, 13) else "1.26.4"
print(f" Select when running this one  NumPy {numpy_version}", flush=True)

packages = [
    "transformer_lens==2.17.0",
    "transformers==4.57.1",
    "huggingface_hub==0.36.0",
    f"numpy=={numpy_version}",
    "pandas==2.2.3",
    "plotly==6.1.2",
    "scipy==1.15.3",
    "scikit-learn==1.6.1",
    "datasets==3.6.0",
    "pyarrow==20.0.0",
    "tqdm",
]
print(" Start installation of dependency; maintain a full log to check. ", flush=True)
try:
    subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        " Reliance on installation failed. Please look above  pip  First in the log  ERROR，"
        " Reruns the block after repair; for the time being, do not run the subsequent block. "
    ) from exc

print("\n Install command successfully executed. Please be on  Colab  . When running again in the menu, "
      " Re-execution 00 path cell in this section, then continue from dependency validation without reinstalling. ", flush=True)


In [ ]:
#  Run this block after restarting: check if the installation is currently in place  Python  Environment. 
import sys
import importlib.metadata as metadata
import importlib.util

print("Python:", sys.version.split()[0])
print(" Interpreter :", sys.executable)
if not ((3, 10) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(" Yes.  Python 3.10–3.13； Please re-implement the installment block after running. ")

expected_versions = {
    "transformer-lens": "2.17.0", "transformers": "4.57.1",
    "huggingface-hub": "0.36.0",
    "numpy": "2.1.3" if sys.version_info[:2] == (3, 13) else "1.26.4",
    "pandas": "2.2.3",
    "plotly": "6.1.2", "scipy": "1.15.3", "scikit-learn": "1.6.1",
    "datasets": "3.6.0", "pyarrow": "20.0.0",
}
problems = []
for package, expected in expected_versions.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    print(f"{package}: {installed if installed is not None else ' not installed '} ( Yes.  {expected})")
    if installed != expected:
        problems.append(f"{package}:  current  {installed}， Yes.  {expected}")

for package in ("torch", "tqdm"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        problems.append(f"{package}:  not installed ")

try:
    torch_version = metadata.version("torch")
    torch_major_minor = tuple(int(part) for part in torch_version.split("+")[0].split(".")[:2])
    if torch_major_minor < (2, 6):
        problems.append(f"torch:  current  {torch_version}， Yes.  >=2.6")
except metadata.PackageNotFoundError:
    pass  #  The record is missing. 
except ValueError:
    problems.append(" Cannot parse  torch  Version, please check the current version  Colab  Run when. ")

if importlib.util.find_spec("transformer_lens") is None:
    problems.append(" current  Python  Cannot find  transformer_lens  module ")
if problems:
    raise RuntimeError(
        " Reliance check failed: \n- " + "\n- ".join(problems)
        + "\n Run the first cell and inspect its complete installation log. After successful installation, restart the runtime, "
          " Go on from this block. "
    )
print(" Reliance on a version check pass; continues to run the third code block. ")


In [ ]:
#  Formal Import: Keep the whole package if there is still an error report here  traceback  In the past few years, many of the worlds leading players have been able to identify specific conflicts. 
import gc, json, math, os, random, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from scipy import stats
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer, utils
from huggingface_hub import notebook_login, get_token, model_info
import importlib.metadata as metadata
from IPython.display import display

PACKAGE_NAMES = ["torch", "transformer-lens", "transformers", "huggingface-hub", "numpy", "pandas", "datasets", "pyarrow", "scipy", "scikit-learn", "plotly"]
PACKAGE_VERSIONS = {name: metadata.version(name) for name in PACKAGE_NAMES}
print(PACKAGE_VERSIONS)
print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print(" Reliance on import success. ")


## 02 · FULL  Configure 

 Only.  aligned/conflict  Two groups. `run_head_search=True`  The new threshold of error in the expression of capacity and pairing is used only after the adoption of the new threshold  v3 train  data searches; not available without prejudice  baseline  And memory spreads. Both characters judge independently. Search is still the same.  3   seed、450  Step, the main conclusion is fixed.  seed=11， Other  seed  For stability diagnosis. 

`run_memory_pipeline=True`  Default run, each  source 3  A follow-up task; this will add model calls but not single input  context  Number of articles. Read-only follow-up tasks  Writer  Output without re-infusing authoritative answers. Task shared before and after  source， Statistics cannot be treated as stand-alone samples. 


In [ ]:
QUICK_MODE = False
MODEL_NAME = 'meta-llama/Llama-3.1-8B'  # BASE， Same as the original experiment. 
MODEL_REVISION = 'd04e592bb4f6aa9cfee91e2e20afa771667e1d4b'
REQUESTED_DTYPE = 'bfloat16'
RUN_NAME = 'agent_entrainment_conflict_v3_full'
FREEZE_AND_RUN_TEST = False
CFG = {
    'model_name': MODEL_NAME, 'dtype': REQUESTED_DTYPE, 'quick_mode': QUICK_MODE,
    'actor_prompt_version': 'verified_profile_and_note_v3',
    'writer_prompt_version': 'authorized_profile_writer_v3',
    'data_design': 'paired_same_target_note_conflict_v3',
    'evaluation_version': 'conflict_v3_full',
    'selection_version': 'complete_behavior_and_writer_joint_v3',
    'conditions': ['aligned', 'conflict'],
    'run_head_search': True, 'run_memory_pipeline': True,
    'data_seed': 1729, 'n_pilot': 24, 'n_train': 120, 'n_dev': 40, 'n_test': 120,
    'search_seeds': [11, 23, 37], 'random_seeds': [101, 202, 303],
    'search_steps': 450, 'batch_size': 1, 'lr': 0.08, 'sparsity': 0.025,
    'temp_start': 1.0, 'temp_end': 0.25, 'clean_weight': 0.1,
    'top_k': [1, 2, 4, 8, 16], 'save_every': 5,
    'hook_kind': 'z', 'use_cache': True, 'context_window': 4096,
    'max_prompt_tokens': 3072, 'actor_max_new_tokens': 128,
    'writer_max_new_tokens': 384, 'max_actions': 6,
    'followup_tasks': 3, 'pipeline_layers': ['end_to_end', 'controlled_decision'],
    'compute_snapshot_margin': True, 'show_progress': True,
    'min_controlled_success': 0.80, 'min_pilot_success': 0.80,
    'max_pilot_format_failure': 0.20, 'min_writer_valid': 0.80,
    'min_writer_clean_correct': 0.80,
    'min_read_conflict_errors': 2, 'min_writer_conflict_errors': 2,
    'bootstrap_reps': 2000,
    'city_candidates': ['Paris', 'Tokyo', 'Rome', 'London', 'Berlin', 'Boston',
        'Chicago', 'Madrid', 'Moscow', 'Seattle', 'Toronto', 'Vienna', 'Dublin',
        'Sydney', 'Austin', 'Denver'],
}
if QUICK_MODE or CFG['conditions'] != ['aligned', 'conflict']:
    raise RuntimeError(' This delivery is fixed. FULL Two conditions; changes in the scope of the experiment require separate protocols and running directories. ')
RUN_DIR = EXPERIMENT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
DEVELOPMENT_COMPLETE = False
DEV_PIPELINE = None
TEST_RESULTS = None
print(' New running directory: ', RUN_DIR)
print(' Every source Twice. context Evaluation. Never two groups. context Merges into an input. ')


## 03 ·  Achieved definition (run all in sequence, foldable) 

 Retain original model generation, query-head `hook_z`  Melting, Gradients  mask  Search and practical tool execution. Model weights are always frozen;  token  Inspections are restricted to diagnostics and searches and do not filter samples of behaviour. 

Writer   `target_correct`  @Amdam: #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 #Jan25 `target_source_correct`  Score provenance only, `target_joint_correct`  Both are required at the same time; `omission`  and  `target_omission`  separately. The main experiment requires only one target profile, but the code supports multiple entities and authorized updates. 

 Formal intervention choice requires retention  aligned  Behaviour / Quality of memory and improvement  conflict  Behavior. Three random matching layers  head  Count and exclude selection  heads。 The threshold is the rule of feasibility of the project and is not the test of salience. 


In [ ]:
# Core infrastructure: exact-prefix token probes, full generation, and search.
import contextlib
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import random
import re
import time
import warnings
import numpy as np
import torch
import torch.nn.functional as F


def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def stable_hash(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def checkpoint_item(stage, key, payload, fn):
    """Only reuse an item produced by exactly this code/model/config/payload."""
    fingerprint = stable_hash({"implementation": IMPLEMENTATION_HASH,
                               "run": RUN_FINGERPRINT, "payload": payload})
    safe_key = re.sub(r"[^A-Za-z0-9_.-]", "_", str(key))[:150]
    path = RUN_DIR / "items" / stage / (safe_key + ".json")
    if path.exists():
        record = json.loads(path.read_text())
        if record["fingerprint"] != fingerprint:
            raise RuntimeError(f"Stale checkpoint: {path}. Use a new RUN_NAME; do not mix experiments.")
        return record["result"]
    result = fn()
    atomic_json(path, {"fingerprint": fingerprint, "payload_hash": stable_hash(payload), "result": result})
    return result


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_heads(heads, n_layers, n_heads):
    result = sorted(set((int(layer), int(head)) for layer, head in heads))
    if any(not (0 <= l < n_layers and 0 <= h < n_heads) for l, h in result):
        raise ValueError("Head coordinates out of bounds")
    return result


def ablation_hooks(model, heads=(), drop_mask=None, hook_kind="z"):
    """Mask query-head outputs, including Llama GQA; never mask KV-head indices."""
    if hook_kind not in {"z", "result"}:
        raise ValueError("hook_kind must be z or result")
    if drop_mask is not None and tuple(drop_mask.shape) != (model.cfg.n_layers, model.cfg.n_heads):
        raise ValueError("Drop mask shape must be [layers, query heads]")
    selected = normalize_heads(heads, model.cfg.n_layers, model.cfg.n_heads)
    if drop_mask is not None and selected:
        raise ValueError("Specify hard heads OR differentiable drop mask")
    by_layer = {}
    for layer, head in selected:
        by_layer.setdefault(layer, []).append(head)
    layers = range(model.cfg.n_layers) if drop_mask is not None else sorted(by_layer)
    hooks = []
    for layer in layers:
        def mask_output(value, hook, layer=layer):
            if drop_mask is not None:
                keep = (1 - drop_mask[layer]).to(device=value.device, dtype=value.dtype)
            else:
                keep = torch.ones(model.cfg.n_heads, device=value.device, dtype=value.dtype)
                keep[by_layer[layer]] = 0
            return value * keep.view(1, 1, -1, 1)
        hooks.append((f"blocks.{layer}.attn.hook_{hook_kind}", mask_output))
    return hooks


class TLBackend:
    def __init__(self, model, cfg):
        self.model = model
        self.tokenizer = model.tokenizer
        self.cfg = dict(cfg)
        self.device = next(model.parameters()).device
        self.hook_kind = cfg.get("hook_kind", "z")
        for parameter in model.parameters():
            parameter.requires_grad_(False)
        model.eval()
        model.set_use_attn_result(self.hook_kind == "result")

    def encode(self, text):
        # One unpadded sequence per forward: no padding-mask/last-position ambiguity.
        return self.tokenizer.encode(text, add_special_tokens=True)

    def tokens(self, text):
        ids = self.encode(text)
        limit = min(int(self.cfg.get("max_prompt_tokens", 3072)), int(self.model.cfg.n_ctx))
        if len(ids) > limit:
            raise ValueError(f"Prompt has {len(ids)} tokens, limit={limit}; evidence is never silently truncated")
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    def one_token_id(self, full_prefix, value):
        before, after = self.encode(full_prefix), self.encode(full_prefix + value)
        if len(after) != len(before) + 1 or after[:-1] != before:
            return None
        token = after[-1]
        if token in set(self.tokenizer.all_special_ids):
            return None
        return token

    def margin(self, snapshot, heads=(), drop_mask=None):
        gold = self.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = self.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            raise ValueError("Snapshot candidates are not two distinct exact-prefix single tokens")
        hooks = ablation_hooks(self.model, heads, drop_mask, self.hook_kind)
        with self.model.hooks(fwd_hooks=hooks):
            logits = self.model(self.tokens(snapshot["prompt"]), return_type="logits")
        # Logit difference equals log P(gold)-log P(distractor).
        return (logits[0, -1, gold] - logits[0, -1, wrong]).float()

    def generate(self, prompt, role="actor", heads=(), max_new_tokens=None, use_cache=True):
        from transformer_lens.past_key_value_caching import HookedTransformerKeyValueCache
        if role not in {"actor", "writer"}:
            raise ValueError("role must be actor or writer")
        budget = int(max_new_tokens or self.cfg[f"{role}_max_new_tokens"])
        prefix = self.tokens(prompt)
        if prefix.shape[1] + budget > int(self.model.cfg.n_ctx):
            raise ValueError("Prompt plus generation budget exceeds model context")
        cache = HookedTransformerKeyValueCache.init_cache(self.model.cfg, self.device, batch_size=1) if use_cache else None
        generated = []
        sequence = prefix
        stop = "max_new_tokens"
        started = time.perf_counter()
        hooks = ablation_hooks(self.model, heads, hook_kind=self.hook_kind)
        # Fresh cache for every call, same intervention for prefill AND decode.
        with torch.inference_mode(), self.model.hooks(fwd_hooks=hooks):
            for step in range(budget):
                tokens = sequence if step == 0 or not use_cache else sequence[:, -1:]
                logits = self.model(tokens, past_kv_cache=cache, return_type="logits")
                next_id = int(logits[0, -1].argmax().item())
                if next_id == self.tokenizer.eos_token_id:
                    stop = "eos"
                    break
                generated.append(next_id)
                sequence = torch.cat([sequence, torch.tensor([[next_id]], device=self.device)], dim=1)
                decoded = self.tokenizer.decode(generated, skip_special_tokens=False)
                parsed, error = parse_json_output(decoded, dict if role == "actor" else list)
                if error is None:
                    stop = "complete_json"
                    break
        text = self.tokenizer.decode(generated, skip_special_tokens=False)
        del cache
        return {"text": text, "token_count": len(generated), "prompt_tokens": int(prefix.shape[1]),
                "seconds": time.perf_counter() - started, "stop_reason": stop}


def eligible_snapshots(backend, cases, role, condition, histories=None):
    retained, excluded = [], []
    for case in cases:
        history = None if histories is None else histories[case["source_id"]]
        snapshot = actor_snapshot(case, condition) if role == "actor" else writer_snapshot(case, condition, history)
        gold = backend.one_token_id(snapshot["prompt"], snapshot["gold"])
        wrong = backend.one_token_id(snapshot["prompt"], snapshot["distractor"])
        if gold is None or wrong is None or gold == wrong:
            excluded.append({"source_id": case["source_id"], "role": role, "condition": condition,
                             "gold_token": gold, "distractor_token": wrong})
        else:
            retained.append(snapshot)
    return retained, excluded


def snapshot_pairs(backend, cases, role, histories_by_condition=None):
    attacks, controls, exclusions = [], [], []
    for case in cases:
        pair = []
        for condition in ("conflict", "aligned"):
            histories = None if histories_by_condition is None else histories_by_condition[condition]
            kept, dropped = eligible_snapshots(backend, [case], role, condition, histories)
            pair.append(kept)
            exclusions.extend(dropped)
        if all(pair):
            attacks.append(pair[0][0]); controls.append(pair[1][0])
    return attacks, controls, exclusions


def gumbel_drop(logits, temperature):
    uniform = torch.rand_like(logits).clamp(1e-6, 1 - 1e-6)
    soft = torch.sigmoid((logits + uniform.log() - torch.log1p(-uniform)) / temperature)
    hard = (soft > 0.5).to(soft.dtype)
    return hard.detach() - soft.detach() + soft


def _save_search(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(record, temporary)
    os.replace(temporary, path)


def train_mask(backend, attacks, controls, cfg, name, seed):
    """Optimize only mask logits. Test episodes never enter this function."""
    if any(s["split"] != "train" for s in attacks + controls):
        raise ValueError("Head search accepts training snapshots only")
    if [s["source_id"] for s in attacks] != [s["source_id"] for s in controls]:
        raise ValueError("Training attack/control pairs are not aligned")
    if not attacks or len(attacks) != len(controls):
        raise ValueError("Training requires nonempty paired snapshots")
    model = backend.model
    shape = (model.cfg.n_layers, model.cfg.n_heads)
    signature = stable_hash({"run": RUN_FINGERPRINT, "code": IMPLEMENTATION_HASH,
                             "attacks": attacks, "controls": controls, "cfg": cfg, "seed": seed, "name": name})
    path = RUN_DIR / "search" / f"{name}-seed{seed}.pt"
    seed_everything(seed)
    with torch.enable_grad():
        logits = torch.nn.Parameter(torch.full(shape, -4.0, device=backend.device, dtype=torch.float32))
        optimizer = torch.optim.AdamW([logits], lr=cfg["lr"], weight_decay=0.0)
        start, history = 0, []
        if path.exists():
            # Only load checkpoints produced by this notebook in your own run directory.
            state = torch.load(path, map_location=backend.device, weights_only=False)
            if state["signature"] != signature:
                raise RuntimeError("Search checkpoint mismatch. Use a new RUN_NAME.")
            logits.data.copy_(state["logits"])
            optimizer.load_state_dict(state["optimizer"])
            start, history = state["step"], state["history"]
            random.setstate(state["python_rng"])
            np.random.set_state(state["numpy_rng"])
            torch.set_rng_state(state["torch_rng"].cpu())
            if torch.cuda.is_available() and state["cuda_rng"] is not None:
                torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda_rng"]])
        with torch.no_grad():
            clean_reference = [float(backend.margin(s).item()) for s in controls]
        for step in range(start, cfg["search_steps"]):
            ratio = step / max(1, cfg["search_steps"] - 1)
            temperature = cfg["temp_start"] * (cfg["temp_end"] / cfg["temp_start"]) ** ratio
            indices = random.sample(range(len(attacks)), min(cfg["batch_size"], len(attacks)))
            optimizer.zero_grad(set_to_none=True)
            total, attack_value, clean_penalty = 0.0, 0.0, 0.0
            # Separate backward per example reduces retained activation memory.
            for index in indices:
                drop = gumbel_drop(logits, temperature)
                attacked_margin = backend.margin(attacks[index], drop_mask=drop)
                clean_margin = backend.margin(controls[index], drop_mask=drop)
                harm = F.relu(torch.tensor(clean_reference[index], device=backend.device) - clean_margin)
                loss = (-attacked_margin + cfg["clean_weight"] * harm + cfg["sparsity"] * drop.sum()) / len(indices)
                loss.backward()
                total += float(loss.detach()); attack_value += float(attacked_margin.detach()) / len(indices)
                clean_penalty += float(harm.detach()) / len(indices)
            if logits.grad is None or not torch.isfinite(logits.grad).all():
                raise RuntimeError("Invalid mask gradient; search did not run correctly")
            grad_norm = float(logits.grad.norm())
            torch.nn.utils.clip_grad_norm_([logits], 5.0)
            optimizer.step()
            history.append({"step": step + 1, "loss": total, "attack_margin": attack_value,
                            "clean_harm": clean_penalty, "temperature": temperature, "grad_norm": grad_norm,
                            "expected_heads": float(torch.sigmoid(logits.detach()).sum())})
            if (step + 1) % cfg["save_every"] == 0 or step + 1 == cfg["search_steps"]:
                _save_search(path, {"signature": signature, "step": step + 1,
                                   "logits": logits.detach(), "optimizer": optimizer.state_dict(), "history": history,
                                   "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                                   "torch_rng": torch.get_rng_state(),
                                   "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None})
                print(f"{name} seed={seed}: {step+1}/{cfg['search_steps']}, margin={attack_value:.3f}, grad={grad_norm:.3g}")
        ranking = torch.argsort(logits.detach().flatten(), descending=True).cpu().tolist()
        heads = [[i // shape[1], i % shape[1]] for i in ranking]
        result = {"name": name, "seed": seed, "ranked_heads": heads,
                  "scores": logits.detach().cpu().tolist(), "history": history, "signature": signature}
    assert all(not p.requires_grad for p in model.parameters())
    atomic_json(RUN_DIR / "search" / f"{name}-seed{seed}.json", result)
    return result


In [ ]:
# Conflict-v3 task data, source authority, strict JSON, and executable Actor tools.
# This section requires only the Python standard library. The model backend is
# defined earlier in the notebook; no model weights are changed by this section.
import copy
import json
import math
import random
import re
import time


TASK_TEMPLATES = {
    "pilot": [
        ("pilot_order", "Create one order for {target}. Try {preferred} first; use {backup} only if the preferred item is unavailable."),
        ("pilot_dispatch", "Arrange a delivery for {target}: preferred item {preferred}, fallback item {backup}. Check availability before reserving."),
    ],
    "train": [
        ("train_purchase", "Complete a purchase for customer {target}. Their first-choice item is {preferred}; the permitted substitute is {backup}."),
        ("train_fulfillment", "Fulfill {target}'s request for {preferred}. If stock prevents that choice, fulfill it with {backup}."),
    ],
    "dev": [
        ("dev_request", "Customer {target} needs an order. Select {preferred} when available and otherwise select {backup}."),
        ("dev_shipment", "Prepare one shipment to {target}. Item priority: {preferred} before {backup}, with substitution allowed only when stock is unavailable."),
    ],
    "test": [
        ("test_checkout", "Finish checkout on behalf of {target}, preferring {preferred}. The only acceptable out-of-stock replacement is {backup}."),
        ("test_procurement", "Procure an item for {target}: obtain {preferred} if possible, else obtain {backup}. Place exactly one resulting order."),
    ],
}

CONDITIONS = ("aligned", "conflict")


def build_cases(split, n, seed, city_values):
    """Generate independent source tasks; all condition variants share a source_id.

    Target values are randomized per source. City labels may be shared across
    splits, but customer IDs and two template families per split are held out.
    This is template-family and entity generalization, not held-out vocabulary.
    Token validity must additionally be checked at each actual model prefix.
    """
    if split not in TASK_TEMPLATES:
        raise ValueError("split must be pilot, train, dev, or test")
    if not isinstance(n, int) or isinstance(n, bool) or n < 0:
        raise ValueError("n must be a nonnegative integer")
    values = list(city_values)
    if len(values) < 4 or len(set(values)) != len(values):
        raise ValueError("city_values requires >=4 distinct strings")
    if any(not isinstance(v, str) or not v or v.strip() != v or
           any(ord(c) < 32 for c in v) for v in values):
        raise ValueError("city_values must be nonempty printable, unpadded strings")
    rng = random.Random(f"{int(seed)}:{split}:order-task-v1")
    cases = []
    for index in range(n):
        target_city, other_city, neutral_city, updated_city = rng.sample(values, 4)
        # One complete 2 x 2 x 2 block per eight sources: template, stock,
        # and memory order vary independently. Partial blocks are incomplete.
        template_id, template = TASK_TEMPLATES[split][(index // 2) % 2]
        source_id = f"{split}-{int(seed)}-{index:05d}"
        preferred_item = f"item-{index % 7:02d}-primary"
        backup_item = f"item-{index % 7:02d}-backup"
        cases.append({
            "source_id": source_id, "split": split,
            "template_id": template_id, "task_template": template,
            "target_name": f"customer-{source_id}-B",
            "other_name": f"customer-{source_id}-A",
            "target_city": target_city, "distractor_city": other_city,
            "neutral_city": neutral_city, "updated_city": updated_city,
            "preferred_item": preferred_item, "backup_item": backup_item,
            "initial_env": {
                "stock": {preferred_item: int(index % 2 == 0), backup_item: 2},
                "reservations": {}, "orders": [],
                "next_reservation": 1, "next_order": 1,
            },
            "position_order": "profile_first" if (index // 4) % 2 == 0 else "note_first",
        })
    return cases


def _preference_record(case, entity, value, suffix):
    return {"kind": "preference", "entity": entity,
            "attribute": "delivery_city", "value": value,
            "source_event": f"{case['source_id']}:{suffix}"}


def expected_city(case, condition):
    """The same authoritative target city is scored in both paired conditions."""
    if condition not in CONDITIONS and condition != "no_memory":
        raise ValueError(f"Unknown condition: {condition}")
    return case["target_city"]


def distractor_city(case, condition):
    """Fixed competing city, even when the aligned note does not contain it."""
    if condition not in CONDITIONS and condition != "no_memory":
        raise ValueError(f"Unknown condition: {condition}")
    return case["distractor_city"]


def memory_for_condition(case, condition):
    """One verified target profile plus one unverified note about that target.

    Aligned/conflict are paired: only the note's city value changes. Source
    kinds, IDs, record order and all other prompt fields stay identical.
    The optional empty-memory input is a dependence diagnostic, not a main arm.
    """
    if condition not in CONDITIONS and condition != "no_memory":
        raise ValueError(f"Unknown condition: {condition}")
    if condition == "no_memory":
        return []
    profile = _preference_record(case, case["target_name"], case["target_city"], "profile:0")
    profile["kind"] = "verified_profile"
    note = _preference_record(
        case, case["target_name"],
        case["target_city"] if condition == "aligned" else case["distractor_city"],
        "note:0",
    )
    note["kind"] = "retrieved_note"
    order = case.get("position_order", "profile_first")
    if order not in ("profile_first", "note_first"):
        raise ValueError(f"Unknown profile/note position order: {order}")
    return [profile, note] if order == "profile_first" else [note, profile]


def authoritative_preferences(history):
    """Gold evidence: latest authorized profile, never a note or observed action.

    Authorization is structural (kind=verified_profile), not inferred from the
    note's prose or source_event name. Sequence orders authorized updates only.
    Returned records use the five-field Writer output schema. Retained memory
    may be read by an Actor but cannot become newly verified Writer gold.
    """
    if not isinstance(history, list):
        raise TypeError("history must be a list of event dictionaries")
    ordered = []
    for index, event in enumerate(history):
        if not isinstance(event, dict):
            raise TypeError(f"history event {index} must be a dictionary")
        if event.get("kind") != "verified_profile":
            continue
        fields = ("entity", "attribute", "value", "source_event")
        if any(not isinstance(event.get(key), str) or not event[key].strip() for key in fields):
            raise ValueError(f"Malformed verified profile at history event {index}")
        if event["attribute"] != "delivery_city":
            raise ValueError("Verified profile attribute must be delivery_city")
        sequence = event.get("sequence", index)
        if (isinstance(sequence, bool) or not isinstance(sequence, (int, float)) or
                not math.isfinite(sequence)):
            raise ValueError("Verified profile sequence must be finite numeric chronology")
        ordered.append((sequence, index, event))
    latest = {}
    for _, _, event in sorted(ordered, key=lambda value: (value[0], value[1])):
        latest[(event["entity"], event["attribute"])] = {
            "kind": "preference", **{key: event[key] for key in ("entity", "attribute", "value", "source_event")}
        }
    return latest


def _json_no_duplicates(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key}")
        result[key] = value
    return result


def parse_json_output(text, expected_type):
    """Strictly parse one JSON object/list; never repair factual values.

    A single enclosing Markdown fence is tolerated. Trailing explanations,
    duplicate keys, NaN/Infinity, and mismatched types are explicit failures.
    """
    if not isinstance(text, str):
        return None, "output_not_text"
    candidate = text.strip()
    if candidate.startswith("```"):
        match = re.fullmatch(r"```(?:json)?\s*\n?(.*?)\n?```", candidate, flags=re.DOTALL | re.IGNORECASE)
        if match is None:
            return None, "invalid_json_fence"
        candidate = match.group(1).strip()
    try:
        value = json.loads(candidate, object_pairs_hook=_json_no_duplicates,
                           parse_constant=lambda token: (_ for _ in ()).throw(ValueError(f"nonfinite JSON: {token}")))
    except (ValueError, TypeError, json.JSONDecodeError) as exc:
        return None, f"invalid_json:{str(exc)[:160]}"
    if not isinstance(value, expected_type):
        return None, f"wrong_json_type:expected_{getattr(expected_type, '__name__', str(expected_type))}"
    return value, None


class OrderEnvironment:
    """A local, deterministic world; it knows inventory, not customer truth."""

    def __init__(self, initial_state):
        self.state = copy.deepcopy(initial_state)
        self.state.setdefault("stock", {})
        self.state.setdefault("reservations", {})
        self.state.setdefault("orders", [])
        self.state.setdefault("next_reservation", 1)
        self.state.setdefault("next_order", 1)
        if any(type(v) is not int or v < 0 for v in self.state["stock"].values()):
            raise ValueError("stock quantities must be nonnegative integers")
        self.tool_log = []

    def snapshot(self):
        return copy.deepcopy(self.state)

    def step(self, action):
        observation = self._execute(action)
        self.tool_log.append({"action": copy.deepcopy(action), "observation": copy.deepcopy(observation)})
        return copy.deepcopy(observation)

    def _execute(self, action):
        if not isinstance(action, dict):
            return {"ok": False, "error": "action_must_be_object"}
        tool = action.get("tool")
        schemas = {
            "check_stock": {"tool", "item"},
            "reserve_item": {"tool", "item"},
            "create_order": {"tool", "customer", "item", "city", "reservation"},
        }
        if not isinstance(tool, str) or tool not in schemas:
            return {"ok": False, "error": "unknown_tool"}
        if set(action) != schemas[tool] or any(not isinstance(v, str) or not v for v in action.values()):
            return {"ok": False, "error": "invalid_arguments", "tool": tool}
        item = action["item"]
        if item not in self.state["stock"]:
            return {"ok": False, "error": "unknown_item", "item": item}
        if tool == "check_stock":
            return {"ok": True, "item": item, "available": self.state["stock"][item]}
        if tool == "reserve_item":
            if self.state["stock"][item] <= 0:
                return {"ok": False, "error": "out_of_stock", "item": item}
            reservation = f"R{self.state['next_reservation']:04d}"
            self.state["next_reservation"] += 1
            self.state["stock"][item] -= 1
            self.state["reservations"][reservation] = {"item": item, "consumed": False}
            return {"ok": True, "item": item, "reservation": reservation}
        reservation = self.state["reservations"].get(action["reservation"])
        if reservation is None or reservation["consumed"] or reservation["item"] != item:
            return {"ok": False, "error": "invalid_reservation"}
        # No oracle city/customer correction: a semantically wrong order can
        # succeed as a tool call. The held-out evaluator scores it afterwards.
        order = {"order_id": f"O{self.state['next_order']:04d}",
                 "customer": action["customer"], "item": item,
                 "city": action["city"], "reservation": action["reservation"]}
        self.state["next_order"] += 1
        self.state["orders"].append(order)
        reservation["consumed"] = True
        return {"ok": True, "order": copy.deepcopy(order)}


def _task_goal(case):
    template = case.get("task_template")
    if template is None:
        template = next((text for _, pairs in TASK_TEMPLATES.items()
                         for key, text in pairs if key == case["template_id"]), None)
    if template is None:
        raise ValueError("Unrecognized task_template/template_id")
    return template.format(target=case["target_name"], preferred=case["preferred_item"], backup=case["backup_item"])


def _actor_protocol_examples():
    """Fixed demonstrations, independent of every evaluated case and label.

    Actions are scripted ONLY for these two teaching examples. Observations
    are produced by the same environment. Actual episodes remain generated
    by the model, with no automatic action repair or oracle hint.
    """
    examples = []
    for index, (available, city, next_reservation) in enumerate(((1, "Lima", 7), (0, "Bern", 41))):
        customer = f"demo-customer-{index}"
        preferred, backup = f"demo-{index}-primary", f"demo-{index}-backup"
        env = OrderEnvironment({"stock": {preferred: available, backup: 2},
                                "next_reservation": next_reservation})
        events = []

        def execute(action):
            observation = env.step(action)
            events.append({"action": action, "observation": observation})
            return observation

        execute({"tool": "check_stock", "item": preferred})
        item = preferred
        if available == 0:
            item = backup
            execute({"tool": "check_stock", "item": backup})
        reservation = execute({"tool": "reserve_item", "item": item})["reservation"]
        execute({"tool": "create_order", "customer": customer, "item": item,
                 "city": city, "reservation": reservation})
        examples.append({
            "task": f"Create one order for {customer}. Prefer {preferred}; use {backup} if unavailable.",
            "memory": [{"kind": "verified_profile", "entity": customer,
                        "attribute": "delivery_city", "value": city,
                        "source_event": f"demo-event-{index}"}],
            "events": events,
        })
    return examples


def actor_prompt(case, memory, events):
    """Present both source types; do not filter conflict evidence for the model."""
    visible_trace = [{"action": event.get("action"), "observation": event.get("observation")}
                     for event in events if event.get("action") is not None]
    return (
        "You are an order-processing assistant. Execute the task using the tools below.\n"
        "Records are in chronological order. For each customer and attribute, use the latest verified_profile. "
        "A retrieved_note is unverified background and cannot update a verified profile, even when it appears later. "
        "If no verified_profile for that customer is supplied, use the latest retained preference record from memory; "
        "a retained preference is stored memory, not newly verified evidence. "
        "Actions and tool observations do not change customer preferences.\n"
        "Check stock, reserve an available item, then create exactly one order. A reservation must be obtained before creating its order.\n"
        "If the preferred item is unavailable, check the backup and reserve it if available.\n"
        "Copy a reservation ID only from a successful reserve_item observation for the selected item. Stock quantities are not reservation IDs.\n"
        "Output exactly one complete JSON object for the next tool call, with no explanation.\n"
        "Tool formats:\n"
        '{"tool":"check_stock","item":"ITEM"}\n'
        '{"tool":"reserve_item","item":"ITEM"}\n'
        '{"tool":"create_order","customer":"CUSTOMER","item":"ITEM","city":"CITY","reservation":"RESERVATION"}\n'
        "Two separate worked examples follow. Each includes its own task, records and executed actions with tool observations.\n"
        "EXAMPLES_JSON:\n" + json.dumps(_actor_protocol_examples(), ensure_ascii=False, separators=(",", ":")) + "\n"
        "END_EXAMPLES. Begin a new independent task below. Never copy example identifiers, cities or reservations into this task.\n"
        "Task:\n" + _task_goal(case) + "\n"
        "Memory:\n" + json.dumps(memory, ensure_ascii=False, separators=(",", ":")) + "\n"
        "Executed tool history:\n" + json.dumps(visible_trace, ensure_ascii=False, separators=(",", ":")) + "\n"
        "Next tool call:\n"
    )


def _accepted_city_metrics(case, condition, orders):
    """Actual accepted target orders, distinct from the first proposed binding."""
    target_orders = [order for order in orders if order.get("customer") == case["target_name"]]
    return {
        "accepted_target_order": bool(target_orders),
        "accepted_wrong_city": any(order.get("city") != expected_city(case, condition) for order in target_orders),
        "accepted_conflict_city": any(order.get("city") == distractor_city(case, condition) for order in target_orders),
        "accepted_wrong_customer": any(order.get("customer") != case["target_name"] for order in orders),
    }


def run_actor(backend, case, memory, heads=(), cfg=None, initial_env=None):
    """Generate complete actions and actually execute them; score only afterwards.

    cfg['condition'] or case['_condition'] identifies the evaluation condition.
    Defaults to conflict (same gold as the paired aligned condition). If an initial
    environment is passed, it is copied; the caller's state is never mutated.
    """
    cfg = {} if cfg is None else dict(cfg)
    condition = cfg.get("condition", case.get("_condition", "conflict"))
    gold = expected_city(case, condition)
    wrong = distractor_city(case, condition)
    initial = copy.deepcopy(case["initial_env"] if initial_env is None else initial_env)
    if isinstance(initial, OrderEnvironment):
        initial = initial.snapshot()
    env = OrderEnvironment(initial)
    max_actions = int(cfg.get("max_actions", 6))
    if max_actions < 1:
        raise ValueError("max_actions must be positive")
    events, actions = [], []
    token_count = prompt_tokens = model_calls = 0
    model_seconds = 0.0
    format_failure = False
    protocol_failure = False
    distractor_action = False
    checked_items = set()
    stock_check_adherence = True
    reservation_attempts = 0
    terminal = "action_budget"
    started = time.perf_counter()
    for step_index in range(max_actions):
        prompt = actor_prompt(case, memory, events)
        output = backend.generate(
            prompt, role="actor", heads=tuple(heads),
            max_new_tokens=cfg.get("actor_max_new_tokens", 128),
            use_cache=cfg.get("use_cache", True),
        )
        model_calls += 1
        raw_output = output.get("text", "")
        token_count += int(output.get("token_count", 0))
        prompt_tokens += int(output.get("prompt_tokens", 0))
        model_seconds += float(output.get("seconds", 0.0))
        action, parse_error = parse_json_output(raw_output, dict)
        event = {"step": step_index, "prompt": prompt, "action": action, "observation": None,
                 "raw_output": raw_output, "parse_error": parse_error,
                 "stop_reason": output.get("stop_reason")}
        if parse_error is not None:
            format_failure = True
            events.append(event)
            terminal = "format_failure"
            break
        actions.append(copy.deepcopy(action))
        if action.get("tool") == "reserve_item":
            reservation_attempts += 1
            stock_check_adherence = stock_check_adherence and isinstance(action.get("item"), str) and action.get("item") in checked_items and case["preferred_item"] in checked_items
        if (action.get("tool") == "create_order" and action.get("customer") == case["target_name"]
                and action.get("city") == wrong):
            distractor_action = True
        observation = env.step(action)
        if action.get("tool") == "check_stock" and observation.get("ok"):
            checked_items.add(action["item"])
        event["observation"] = observation
        events.append(event)
        if observation.get("error") in {"action_must_be_object", "unknown_tool", "invalid_arguments"}:
            protocol_failure = True
            format_failure = True
            terminal = "protocol_failure"
            break
        if action.get("tool") == "create_order" and observation.get("ok"):
            terminal = "order_created"
            break
    new_orders = env.state["orders"][len(initial.get("orders", [])):]
    preferred_available = initial["stock"].get(case["preferred_item"], 0) > 0
    gold_item = case["preferred_item"] if preferred_available else case["backup_item"]
    success = bool(len(new_orders) == 1 and new_orders[0]["customer"] == case["target_name"]
                   and new_orders[0]["item"] == gold_item and new_orders[0]["city"] == gold)
    result = {
        "source_id": case["source_id"], "split": case["split"], "condition": condition,
        "template_id": case["template_id"], "preferred_available": preferred_available,
        "position_order": case["position_order"],
        "success": success, "task_success": success,
        "distractor_action": distractor_action,
        **_accepted_city_metrics(case, condition, new_orders),
        "format_failure": format_failure, "protocol_failure": protocol_failure,
        "stock_check_adherence": bool(stock_check_adherence and reservation_attempts > 0),
        "budget_exhausted": terminal == "action_budget", "terminal": terminal,
        "actions": actions, "events": events, "final_state": env.snapshot(),
        "initial_state": initial, "token_count": token_count, "prompt_tokens": prompt_tokens,
        "seconds": time.perf_counter() - started, "model_seconds": model_seconds,
        "model_calls": model_calls, "action_count": len(actions),
        "accepted_order_count": len(new_orders),
        "gold_city": gold, "distractor_city": wrong, "gold_item": gold_item,
        "memory": copy.deepcopy(memory),
    }
    result["metrics"] = {k: result[k] for k in (
        "task_success", "distractor_action", "format_failure", "protocol_failure",
        "stock_check_adherence", "budget_exhausted", "action_count", "accepted_order_count",
        "accepted_wrong_city", "accepted_conflict_city", "accepted_target_order", "accepted_wrong_customer")}
    result["evaluation_layer"] = "end_to_end"
    layered = score_actor_layers(case, result)
    result.update(layered)
    result["metrics"].update({k: v for k, v in layered.items() if isinstance(v, (bool, int, float))})
    return result


def frozen_history(case, condition, actor_result=None):
    """Freeze source-tagged profiles/notes and, optionally, observed execution.

    A wrong note remains unverified; a wrong action remains an observed action.
    Neither becomes a verified profile, and trajectories are never invented.
    """
    if actor_result is not None and (actor_result.get("source_id") != case["source_id"]
                                     or actor_result.get("condition") != condition):
        raise ValueError("Frozen history Actor result does not match source/condition")
    history = []
    for record in memory_for_condition(case, condition):
        event = copy.deepcopy(record)
        event["sequence"] = len(history)
        history.append(event)
    if actor_result is not None:
        for event in actor_result["events"]:
            step = event["step"]
            if event.get("action") is not None:
                history.append({"kind": "observed_action", "sequence": len(history),
                                "source_event": f"{case['source_id']}:action:{step}",
                                "action": copy.deepcopy(event["action"])})
            else:
                history.append({"kind": "observed_output", "sequence": len(history),
                                "source_event": f"{case['source_id']}:output:{step}",
                                "text": event.get("raw_output", ""),
                                "parse_error": event.get("parse_error")})
            if event.get("observation") is not None:
                history.append({"kind": "tool_observation", "sequence": len(history),
                                "source_event": f"{case['source_id']}:observation:{step}",
                                "observation": copy.deepcopy(event["observation"])})
    return history


def prepare_legal_decision_state(case):
    """Execute tool prerequisites only; independent of memory and city labels."""
    env = OrderEnvironment(case["initial_env"])
    events = []
    preferred = case["preferred_item"]
    check = {"tool": "check_stock", "item": preferred}
    observation = env.step(check)
    events.append({"action": check, "observation": observation})
    item = preferred
    if observation["available"] <= 0:
        item = case["backup_item"]
        check = {"tool": "check_stock", "item": item}
        events.append({"action": check, "observation": env.step(check)})
    reserve = {"tool": "reserve_item", "item": item}
    reservation_observation = env.step(reserve)
    if not reservation_observation.get("ok"):
        raise ValueError("Snapshot requires at least one available allowed item")
    events.append({"action": reserve, "observation": reservation_observation})
    return {"state": env.snapshot(), "events": events, "item": item,
            "reservation": reservation_observation["reservation"]}


def actor_snapshot(case, condition):
    """Teacher-forced city-token probe, separate from complete-action tests."""
    prepared = prepare_legal_decision_state(case)
    item, events = prepared["item"], prepared["events"]
    prefix_fields = {"tool": "create_order", "customer": case["target_name"],
                     "item": item, "reservation": prepared["reservation"]}
    action_prefix = json.dumps(prefix_fields, ensure_ascii=False, separators=(",", ":"))[:-1] + ',"city":"'
    prompt = actor_prompt(case, memory_for_condition(case, condition), events) + action_prefix
    return {"source_id": case["source_id"], "split": case["split"],
            "role": "actor", "prompt": prompt,
            "gold": expected_city(case, condition),
            "distractor": distractor_city(case, condition), "condition": condition,
            "template_id": case["template_id"], "action_prefix": action_prefix,
            "probe_state": prepared["state"], "probe_events": events,
            "snapshot_origin": "constructed_legal_prefix"}


def first_order_binding(case, events, condition):
    """First parsed create_order attempt, even if execution fails. All episodes count.

    This measures an observable proposed binding, not hidden model knowledge.
    Missing/malformed outputs are never repaired and never leave the denominator.
    """
    first = next((e for e in events if isinstance(e.get("action"), dict)
                  and e["action"].get("tool") == "create_order"), None)
    category = "no_order_attempt"
    if first is not None:
        action = first["action"]
        if any(not isinstance(action.get(k), str) or not action[k].strip() for k in ("customer", "city")):
            category = "invalid_binding_fields"
        elif action["customer"] != case["target_name"]:
            category = "wrong_customer"
        elif action["city"] == expected_city(case, condition):
            category = "correct_binding"
        elif action["city"] == distractor_city(case, condition):
            category = "distractor_binding"
        else:
            category = "other_city"
    categories = ("no_order_attempt", "invalid_binding_fields", "wrong_customer",
                  "correct_binding", "distractor_binding", "other_city")
    return {"binding_category": category, "order_attempted": first is not None,
            "first_order_step": first.get("step") if first else None,
            **{"binding_" + name: category == name for name in categories},
            "binding_correct": category == "correct_binding",
            "binding_distractor": category == "distractor_binding",
            "binding_conflict": category == "distractor_binding"}


def score_actor_layers(case, result):
    """Project-specific subgoals and prerequisite subsequence, inspired by
    AgentBoard/BFCL. Extra legal steps are permitted; a literal path is not forced.
    This is a trace diagnostic, not a causal attribution algorithm.
    """
    events = result["events"]
    preferred, chosen = case["preferred_item"], result["gold_item"]
    checked = {}
    witnessed_preferred = witnessed_chosen = reserved_chosen = False
    grounded_reservations = set()
    prerequisite_path = False
    first_error_step, first_error = None, None
    for i, event in enumerate(events):
        action, obs = event.get("action") or {}, event.get("observation") or {}
        error = event.get("parse_error") or obs.get("error")
        if error and first_error is None:
            first_error_step, first_error = i, error
        tool, item = action.get("tool"), action.get("item")
        if tool == "check_stock" and obs.get("ok"):
            checked[item] = obs["available"]
            witnessed_preferred |= item == preferred
            witnessed_chosen |= item == chosen and obs["available"] > 0
        if tool == "reserve_item" and obs.get("ok") and item == chosen:
            reserved_chosen = True
            priority_ok = (checked.get(preferred, -1) > 0 if chosen == preferred
                           else checked.get(preferred, -1) == 0)
            if priority_ok and checked.get(chosen, 0) > 0:
                grounded_reservations.add(obs["reservation"])
        if tool == "create_order" and obs.get("ok") and action.get("item") == chosen:
            prerequisite_path |= action.get("reservation") in grounded_reservations
    success = bool(result["task_success"])
    # Four independently observable milestones; their average is descriptive.
    progress = (int(witnessed_preferred) + int(witnessed_chosen) + int(reserved_chosen) + int(success)) / 4
    return {**first_order_binding(case, events, result["condition"]),
            "checked_preferred": witnessed_preferred, "checked_selected_available": witnessed_chosen,
            "reserved_selected": reserved_chosen, "subgoal_progress": progress,
            "prerequisite_path_valid": prerequisite_path,
            "strict_task_success": success and prerequisite_path,
            "first_tool_error_step": first_error_step, "first_tool_error": first_error,
            "binding_scoring": "first_parsed_order_attempt_all_sources"}


def run_controlled_actor(backend, case, memory, heads=(), cfg=None, initial_env=None):
    """One freely generated complete action after real prerequisite execution.

    No answer/action prefix is supplied. Prepared events are setup, not model
    actions or Writer histories. The original stock determines the target item.
    """
    cfg = {} if cfg is None else dict(cfg)
    case = copy.deepcopy(case)
    if initial_env is not None:
        case["initial_env"] = (initial_env.snapshot() if isinstance(initial_env, OrderEnvironment)
                               else copy.deepcopy(initial_env))
    condition = cfg.get("condition", case.get("_condition", "conflict"))
    prepared = prepare_legal_decision_state(case)
    env = OrderEnvironment(prepared["state"])
    prompt = actor_prompt(case, memory, prepared["events"])
    output = backend.generate(prompt, role="actor", heads=tuple(heads),
                              max_new_tokens=cfg.get("actor_max_new_tokens", 128),
                              use_cache=cfg.get("use_cache", True))
    action, parse_error = parse_json_output(output.get("text", ""), dict)
    observation = env.step(action) if parse_error is None else None
    protocol_error = bool(observation and observation.get("error") in
                          {"action_must_be_object", "unknown_tool", "invalid_arguments"})
    event = {"step": 0, "prompt": prompt, "raw_output": output.get("text", ""),
             "action": action, "observation": observation, "parse_error": parse_error,
             "stop_reason": output.get("stop_reason")}
    new_orders = env.state["orders"][len(prepared["state"]["orders"]):]
    success = bool(len(new_orders) == 1 and new_orders[0]["customer"] == case["target_name"]
                   and new_orders[0]["item"] == prepared["item"]
                   and new_orders[0]["city"] == expected_city(case, condition))
    result = {"source_id": case["source_id"], "split": case["split"], "condition": condition,
              "evaluation_layer": "controlled_decision", "setup_origin": "executed_tool_prerequisites",
              "template_id": case["template_id"], "position_order": case["position_order"],
              "preferred_available": case["initial_env"]["stock"][case["preferred_item"]] > 0,
              "task_success": success, "success": success, "decision_action_success": success,
              "format_failure": parse_error is not None or protocol_error,
              "protocol_failure": protocol_error,
              "tool_execution_valid": bool(observation and observation.get("ok")),
              "distractor_action": bool(action and action.get("tool") == "create_order"
                                         and action.get("customer") == case["target_name"]
                                         and action.get("city") == distractor_city(case, condition)),
              **_accepted_city_metrics(case, condition, new_orders),
              "events": [event], "setup_events": prepared["events"],
              "setup_state": prepared["state"], "final_state": env.snapshot(),
              "initial_state": case["initial_env"], "gold_item": prepared["item"],
              "gold_city": expected_city(case, condition), "memory": copy.deepcopy(memory),
              "accepted_order_count": len(new_orders), "model_calls": 1,
              **{key: output.get(key, 0) for key in ("token_count", "prompt_tokens", "seconds")},
              **first_order_binding(case, [event], condition)}
    result["metrics"] = {k: v for k, v in result.items() if isinstance(v, (bool, int, float))}
    return result


def run_downstream_actor(backend, case, memory, heads=(), cfg=None, initial_env=None):
    """Run only retained Writer memory; invalid memory is an explicit failed source.

    This function never reconstructs a profile from evaluator truth. An invalid
    Writer output is an upstream input failure, not an Actor JSON-generation
    failure, and consumes zero model calls.
    """
    cfg = {} if cfg is None else dict(cfg)
    layer = cfg.get("evaluation_layer", "end_to_end")
    if layer not in ("end_to_end", "controlled_decision"):
        raise ValueError("Unknown evaluation layer")
    error = None
    required_keys = {"kind", "entity", "attribute", "value", "source_event"}
    if not isinstance(memory, list):
        error = "writer_memory_not_a_list"
    else:
        for index, record in enumerate(memory):
            if (not isinstance(record, dict) or set(record) != required_keys or
                    record.get("kind") != "preference" or record.get("attribute") != "delivery_city" or
                    any(not isinstance(record.get(key), str) or not record[key].strip()
                        for key in ("entity", "attribute", "value", "source_event"))):
                error = f"invalid_writer_memory_record:{index}"
                break
    if error is not None:
        condition = cfg.get("condition", case.get("_condition", "conflict"))
        initial = case["initial_env"] if initial_env is None else initial_env
        initial = initial.snapshot() if isinstance(initial, OrderEnvironment) else copy.deepcopy(initial)
        preferred_available = initial["stock"].get(case["preferred_item"], 0) > 0
        result = {
            "source_id": case["source_id"], "split": case["split"], "condition": condition,
            "evaluation_layer": layer, "template_id": case["template_id"],
            "position_order": case["position_order"], "preferred_available": preferred_available,
            "success": False, "task_success": False, "strict_task_success": False,
            "decision_action_success": False, "input_memory_failure": True,
            "memory_format_failure": True, "format_failure": False, "protocol_failure": False,
            "memory_error": error, "terminal": "invalid_writer_memory", "skipped_model_call": True,
            "tool_execution_valid": False, "distractor_action": False,
            "stock_check_adherence": False, "budget_exhausted": False,
            "checked_preferred": False, "checked_selected_available": False, "reserved_selected": False,
            "prerequisite_path_valid": False, "subgoal_progress": 0.0,
            "accepted_order_count": 0, "action_count": 0, "model_calls": 0,
            "token_count": 0, "prompt_tokens": 0, "seconds": 0.0, "model_seconds": 0.0,
            "events": [], "actions": [], "setup_events": [], "memory": copy.deepcopy(memory),
            "initial_state": initial, "final_state": copy.deepcopy(initial),
            "gold_item": case["preferred_item"] if preferred_available else case["backup_item"],
            "gold_city": expected_city(case, condition), "distractor_city": distractor_city(case, condition),
            "first_tool_error_step": None, "first_tool_error": None,
            **first_order_binding(case, [], condition), **_accepted_city_metrics(case, condition, []),
        }
        result["metrics"] = {key: value for key, value in result.items() if isinstance(value, (bool, int, float))}
        return result
    runner = run_controlled_actor if layer == "controlled_decision" else run_actor
    result = runner(backend, case, copy.deepcopy(memory), heads=heads, cfg=cfg, initial_env=initial_env)
    result["input_memory_failure"] = False
    result["memory_format_failure"] = False
    result["metrics"].update(input_memory_failure=False, memory_format_failure=False)
    return result


In [ ]:
"""Conflict-v3 Writer and unconditional memory propagation.

This cell is embedded verbatim in the standalone Colab notebook. It uses the
Actor, checkpoint and model helpers defined in earlier notebook cells.
"""
import copy
import math


def _latest_preference_records(history):
    """Only authorized profiles establish scoring truth; recency is secondary."""
    return authoritative_preferences(history)


def _memory_schema_error(memory):
    if not isinstance(memory, list):
        return "memory must be a JSON list"
    required = {"kind", "entity", "attribute", "value", "source_event"}
    for index, record in enumerate(memory):
        if not isinstance(record, dict) or set(record) != required:
            return f"record {index}: keys must be exactly {sorted(required)}"
        if record["kind"] != "preference":
            return f"record {index}: kind must be preference"
        if record["attribute"] != "delivery_city":
            return f"record {index}: attribute must be delivery_city"
        if any(not isinstance(record[key], str) or not record[key].strip()
               for key in ("entity", "attribute", "value", "source_event")):
            return f"record {index}: preference fields must be nonempty strings"
    return None


def _writer_presented_history(history):
    """Preserve observed order and provenance, removing only sequence fields."""
    if not isinstance(history, list):
        raise TypeError("history must be a list")
    presented = copy.deepcopy(history)
    previous_sequence = None
    for index, event in enumerate(presented):
        if not isinstance(event, dict):
            raise TypeError(f"history event {index} must be a dictionary")
        sequence = event.get("sequence", index)
        if (isinstance(sequence, bool) or not isinstance(sequence, (int, float))
                or not math.isfinite(sequence)):
            raise ValueError(f"history event {index}: sequence must be finite numeric")
        if previous_sequence is not None and sequence < previous_sequence:
            raise ValueError("History order conflicts with sequence order")
        previous_sequence = sequence
        event.pop("sequence", None)
    return presented


def _writer_protocol_examples():
    """Fixed independent examples cover source authority and authorized updating."""
    def profile(entity, city, event):
        return {"kind": "verified_profile", "entity": entity,
                "attribute": "delivery_city", "value": city,
                "source_event": event}

    first = profile("demo-writer-client-a", "DemoHarbor", "demo-profile-a")
    note = {**first, "kind": "retrieved_note", "value": "DemoMeadow",
            "source_event": "demo-unverified-note-a"}
    old = profile("demo-writer-client-b", "DemoHarbor", "demo-profile-b-old")
    updated = profile("demo-writer-client-b", "DemoSummit", "demo-profile-b-update")
    late_note = {**old, "kind": "retrieved_note", "source_event": "demo-late-note-b"}
    return [
        {"name": "A retrieved note and an observed order do not update a verified profile",
         "target_name": first["entity"],
         "events": [first, note,
                    {"kind": "observed_action", "source_event": "demo-action-a",
                     "action": {"tool": "create_order", "customer": first["entity"],
                                "city": "DemoMeadow", "item": "demo-item-a",
                                "reservation": "demo-reservation-a"}},
                    {"kind": "tool_observation", "source_event": "demo-observation-a",
                     "observation": {"ok": True, "city": "DemoMeadow"}}],
         "memory": [{**first, "kind": "preference"}]},
        {"name": "A later verified profile updates value and source; a later note does not",
         "target_name": old["entity"],
         "events": [old, updated, late_note],
         "memory": [{**updated, "kind": "preference"}]},
    ]


def writer_prompt(case, history):
    """Generate whole memory from visible evidence; no evaluator truth is appended."""
    presented = _writer_presented_history(history)
    examples = "\n\n".join(
        "EXAMPLE " + str(index + 1) + ": " + example["name"] + "\n"
        + "Example target: " + canonical_json(example["target_name"]) + "\n"
        + "Example events:\n" + canonical_json(example["events"]) + "\n"
        + "Example complete output:\n" + canonical_json(example["memory"])
        for index, example in enumerate(_writer_protocol_examples())
    )
    return (
        "You are the memory writer for an order-processing assistant.\n"
        "Read the observed events and output ONLY one JSON list of preference records.\n"
        'Each record has exactly these fields: {"kind":"preference",'
        '"entity":"customer name","attribute":"delivery_city",'
        '"value":"city","source_event":"verified profile event id"}.\n'
        "Only verified_profile events authorize a delivery-city preference. "
        "For each entity and attribute, use the latest verified_profile event, "
        "and copy its value and source_event. Events are listed oldest to newest. "
        "A retrieved_note may be inaccurate: it never updates a verified profile, "
        "even when the note appears later or mentions the same customer. "
        "Actions and tool observations report what happened, not what the customer "
        "prefers. Do not infer a preference from an order destination. "
        "Metadata and previous_memory records are not new verified evidence.\n"
        "Keep distinct customers separate. Output at most one record per entity "
        "and attribute. Put the requested target first if present, then all other "
        "verified preferences. Retain every required verified preference; do not "
        "discard a profile because a note disagrees with it.\n"
        "Do not copy demo records. Do not add explanations, extra fields, or "
        "Markdown fences. Stop after the closing ] of the single list.\n\n"
        + examples
        + "\nEND_WRITER_EXAMPLES.\n\nCURRENT_TASK\nTarget customer: "
        + canonical_json(case["target_name"])
        + "\nEVENTS_JSON:\n" + canonical_json(presented) + "\nMEMORY_JSON:\n"
    )


def score_memory(case, history, memory, valid):
    """Separate value, source, joint correctness, omission and protocol quality.

    References come only from verified_profile events. `omission` means any
    required entity/attribute is missing; `target_omission` is target-specific.
    A correct value with an unverified note's source fails target_joint_correct.
    No output is repaired or removed from the planned source denominator.
    """
    reference = _latest_preference_records(history)
    valid = bool(valid) and _memory_schema_error(memory) is None
    records = memory if valid else []
    by_key = {}
    for record in records:
        by_key.setdefault((record["entity"], record["attribute"]), []).append(record)
    target_key = (case["target_name"], "delivery_city")
    target_reference = reference.get(target_key)
    target_records = by_key.get(target_key, [])
    has_reference = bool(reference)
    condition = case.get("_condition", "conflict")
    wrong = distractor_city(case, condition)
    missing = sum(key not in by_key for key in reference)
    value_correct = (valid and len(target_records) == 1
                     and target_records[0]["value"] == target_reference["value"]
                     if target_reference is not None else None)
    source_correct = (valid and len(target_records) == 1
                      and target_records[0]["source_event"] == target_reference["source_event"]
                      if target_reference is not None else None)
    conflict_error = (any(record["value"] == wrong for record in target_records)
                      and wrong != target_reference["value"]
                      if target_reference is not None else None)
    unsupported = (sum(key not in reference or record["value"] != reference[key]["value"]
                       for key, entries in by_key.items() for record in entries)
                   if has_reference else None)
    source_mismatch = (sum(key in reference and record["source_event"] != reference[key]["source_event"]
                           for key, entries in by_key.items() for record in entries)
                       if has_reference else None)
    correct = (sum(len(by_key.get(key, [])) == 1 and by_key[key][0]["value"] == gold["value"]
                   for key, gold in reference.items()) if has_reference else None)
    joint_correct = (sum(len(by_key.get(key, [])) == 1
                         and by_key[key][0]["value"] == gold["value"]
                         and by_key[key][0]["source_event"] == gold["source_event"]
                         for key, gold in reference.items()) if has_reference else None)
    return {
        "target_correct": value_correct,
        "target_source_correct": source_correct,
        "target_joint_correct": (bool(value_correct and source_correct)
                                 if target_reference is not None else None),
        "target_conflict_error": conflict_error,
        "target_distractor_error": conflict_error,  # Compatibility alias, same definition.
        "omission": bool(missing) if has_reference else None,
        "target_omission": not target_records if target_reference is not None else None,
        "missing_count": missing if has_reference else None,
        "unsupported_count": unsupported,
        "source_mismatch_count": source_mismatch,
        "duplicate_count": sum(max(0, len(entries) - 1) for entries in by_key.values()),
        "format_failure": not valid,
        "required_count": len(reference), "record_count": len(records),
        "correct_count": correct, "joint_correct_count": joint_correct,
        "whole_memory_correct": (valid and len(records) == len(reference)
                                 and joint_correct == len(reference)
                                 if has_reference else None),
        "reference_available": target_reference is not None,
        "reference_policy": "latest_authorized_verified_profile_v1",
    }


def run_writer(backend, case, history, heads=(), cfg=None):
    cfg = {} if cfg is None else dict(cfg)
    evidence = copy.deepcopy(history)
    before = stable_hash(evidence)
    prompt = writer_prompt(case, evidence)
    generation = backend.generate(
        prompt, role="writer", heads=tuple(tuple(head) for head in heads),
        max_new_tokens=cfg.get("writer_max_new_tokens", 384),
        use_cache=cfg.get("use_cache", True),
    )
    raw = generation.get("text", "")
    memory, parse_error = parse_json_output(raw, expected_type=list)
    schema_error = None if parse_error else _memory_schema_error(memory)
    error = parse_error or schema_error
    valid = error is None
    memory = memory if valid else None
    if stable_hash(history) != before or stable_hash(evidence) != before:
        raise RuntimeError("Writer changed its fixed observed history")
    return {
        "memory": memory, "valid": valid, "raw": raw, "error": error,
        "metrics": score_memory(case, evidence, memory, valid),
        **{key: generation.get(key, 0) for key in ("token_count", "prompt_tokens", "seconds")},
        "stop_reason": generation.get("stop_reason"),
        "history_hash": before, "prompt_hash": stable_hash(prompt),
        "memory_origin": "model_generated_from_controlled_context",
    }


def writer_snapshot(case, condition, history=None):
    """Teacher-forced token probe; full free-generation outcomes remain primary."""
    if condition not in CONDITIONS:
        raise ValueError(f"Unknown condition: {condition}")
    history = frozen_history(case, condition) if history is None else copy.deepcopy(history)
    reference = _latest_preference_records(history)
    key = (case["target_name"], "delivery_city")
    if key not in reference:
        raise ValueError("Writer probe requires a verified target profile")
    value_prefix = ('[{"kind":"preference","entity":'
                    + canonical_json(case["target_name"])
                    + ',"attribute":"delivery_city","value":"')
    return {
        "source_id": case["source_id"], "split": case["split"], "role": "writer",
        "prompt": writer_prompt(case, history) + value_prefix,
        "gold": reference[key]["value"], "distractor": distractor_city(case, condition),
        "condition": condition, "history_hash": stable_hash(history),
        "template_id": case.get("template_id"), "snapshot_origin": "constructed_writer_value_prefix",
    }


def _writer_progress(values, description, cfg):
    progress = globals().get("tqdm")
    if cfg.get("show_progress", True) and callable(progress):
        return progress(values, desc=description, leave=False)
    return values


def _cached_writer_result(backend, case, history, heads, cfg):
    """Generation identity excludes display labels, allowing exact reuse in pipeline."""
    condition = case.get("_condition", "conflict")
    if condition not in CONDITIONS:
        raise ValueError(f"Unknown Writer condition: {condition}")
    writer_cfg = dict(cfg, condition=condition)
    writer_cfg.pop("evaluation_layer", None)  # Downstream layer cannot change a Writer.
    heads = [list(head) for head in heads]
    history_hash = stable_hash(history)
    prompt = writer_prompt(case, history)
    payload = {"case": case, "history": history, "heads": heads, "cfg": writer_cfg,
               "prompt": prompt, "version": "conflict_v3_authorized_writer"}
    key = f'{case["source_id"]}-{condition}-' + stable_hash(payload)[:20]
    result = checkpoint_item(
        "writer_evaluation", key, payload,
        lambda: run_writer(backend, copy.deepcopy(case), copy.deepcopy(history), heads, writer_cfg),
    )
    if result.get("history_hash") != history_hash or result.get("prompt_hash") != stable_hash(prompt):
        raise RuntimeError("Writer checkpoint does not match current history/prompt")
    if stable_hash(history) != history_hash:
        raise RuntimeError("Writer evaluation changed its input history")
    return copy.deepcopy(result)


def collect_fixed_histories(backend, cases, conditions, cfg, actor_rows=None):
    """Freeze actual untreated E2E histories; optionally reuse already evaluated rows.

    Supplying actor_rows never launches Actor inference. Missing/duplicate rows
    are errors rather than silently dropping sources. Without rows, exactly the
    required untreated Actor runs are checkpointed and then frozen.
    """
    conditions = tuple(conditions)
    if not conditions or any(condition not in CONDITIONS for condition in conditions):
        raise ValueError("Histories require supported conditions")
    source_ids = [case["source_id"] for case in cases]
    if not source_ids or len(set(source_ids)) != len(source_ids):
        raise ValueError("Histories require unique, nonempty source cases")
    expected = {(source, condition) for source in source_ids for condition in conditions}
    indexed = {}
    if actor_rows is not None:
        for row in actor_rows:
            if row.get("intervention", "none") != "none" or row.get("evaluation_layer") != "end_to_end":
                continue
            key = (row.get("source_id"), row.get("condition"))
            if key not in expected:
                continue
            if key in indexed:
                raise ValueError(f"Duplicate untreated Actor trajectory: {key}")
            indexed[key] = row
        if set(indexed) != expected:
            raise ValueError(f"Missing untreated Actor trajectories: {sorted(expected - set(indexed))}")
    histories = {condition: {} for condition in conditions}
    jobs = [(case, condition) for case in cases for condition in conditions]
    for original_case, condition in _writer_progress(jobs, " Freezing real experiences ", cfg):
        case = copy.deepcopy(original_case)
        case["_condition"] = condition
        source_id = case["source_id"]
        if actor_rows is None:
            actor_cfg = dict(cfg, condition=condition)
            memory = memory_for_condition(case, condition)
            payload = {"case": case, "memory": memory, "heads": [], "cfg": actor_cfg,
                       "purpose": "conflict_v3_fixed_writer_history"}
            key = f"{source_id}-{condition}-" + stable_hash(payload)[:20]
            actor_result = checkpoint_item(
                "fixed_history_actors", key, payload,
                lambda: run_actor(backend, copy.deepcopy(case), copy.deepcopy(memory),
                                  heads=(), cfg=actor_cfg),
            )
        else:
            actor_result = indexed[source_id, condition]
            if actor_result.get("memory") != memory_for_condition(case, condition):
                raise ValueError("Actor trajectory context does not match the current case/condition")
        if actor_result.get("source_id") != source_id or actor_result.get("condition") != condition:
            raise ValueError("Actor trajectory provenance does not match the requested history")
        histories[condition][source_id] = frozen_history(case, condition, actor_result)
    return histories


def evaluate_writer(backend, cases, histories, head_sets, cfg, conditions=CONDITIONS):
    """Full generated memories for every planned source, including invalid outputs."""
    if not isinstance(head_sets, dict) or not head_sets:
        raise ValueError("head_sets must be a nonempty mapping")
    source_ids = [case["source_id"] for case in cases]
    if not source_ids or len(source_ids) != len(set(source_ids)):
        raise ValueError("Writer evaluation needs unique nonempty source cases")
    if not conditions or any(condition not in CONDITIONS for condition in conditions):
        raise ValueError("Unknown Writer conditions")
    rows = []
    jobs = [(case, condition) for case in cases for condition in conditions]
    for original_case, condition in _writer_progress(jobs, "Writer  Full memory. ", cfg):
        case = copy.deepcopy(original_case)
        case["_condition"] = condition
        history = copy.deepcopy(histories[condition][case["source_id"]])
        for label, heads in head_sets.items():
            result = _cached_writer_result(backend, case, history, heads, cfg)
            rows.append({
                "source_id": case["source_id"], "split": case["split"],
                "condition": condition, "intervention": str(label),
                "heads": [list(head) for head in heads],
                **result["metrics"], "valid": result["valid"],
                "token_count": result.get("token_count", 0), "seconds": result.get("seconds", 0.0),
                "history_hash": result["history_hash"], "writer_result": result,
            })
    return rows


def build_memory_pair(backend, case, history, write_heads, cfg):
    """M0 always exists; M1 exists only for an actual nonempty Writer intervention."""
    before = stable_hash(history)
    pair = {"source_id": case["source_id"], "split": case["split"],
            "condition": case.get("_condition", "conflict"), "tau_hash": before,
            "write_heads": [list(head) for head in write_heads],
            "M0": _cached_writer_result(backend, case, history, (), cfg)}
    if write_heads:
        pair["M1"] = _cached_writer_result(backend, case, history, write_heads, cfg)
        if pair["M0"]["prompt_hash"] != pair["M1"]["prompt_hash"]:
            raise RuntimeError("Writer arms received different prompts")
    if stable_hash(history) != before:
        raise RuntimeError("Memory pairing changed the fixed history")
    return pair


def _followup_case_and_environment(case, index):
    """Matched fresh order tasks; only retained memory carries into each followup."""
    if type(index) is not int or index < 0:
        raise ValueError("followup index must be a nonnegative integer")
    followup = copy.deepcopy(case)
    initial = copy.deepcopy(case["initial_env"])
    initial.update(reservations={}, orders=[], next_reservation=1, next_order=1)
    if index % 3 == 2:
        followup["preferred_item"], followup["backup_item"] = followup["backup_item"], followup["preferred_item"]
    initial["stock"][followup["preferred_item"]] = 0 if index % 3 == 1 else 1 + index // 3
    initial["stock"][followup["backup_item"]] = 1 + index
    followup["initial_env"] = copy.deepcopy(initial)
    followup["followup_id"] = f'{case["source_id"]}:followup:{index}'
    return followup, initial


def evaluate_factorial(backend, case, pair, read_heads, cfg):
    """Evaluate only arms that actually exist, never an artificial empty-head 2x2."""
    if pair["source_id"] != case["source_id"]:
        raise ValueError("Memory pair and case have different source IDs")
    for label in ("M0", "M1"):
        if label in pair and pair[label]["history_hash"] != pair["tau_hash"]:
            raise ValueError("Writer memories do not share the fixed source history")
    count = cfg.get("followup_tasks", 3)
    if type(count) is not int or count < 1:
        raise ValueError("followup_tasks must be a positive integer")
    layer = cfg.get("evaluation_layer", "end_to_end")
    if layer not in ("end_to_end", "controlled_decision"):
        raise ValueError("Unsupported propagation evaluation layer")
    arms = [("baseline", "M0", (), 0, 0)]
    if "M1" in pair:
        arms.append(("writer_only", "M1", (), 1, 0))
    if read_heads:
        arms.append(("actor_only", "M0", read_heads, 0, 1))
    if read_heads and "M1" in pair:
        arms.append(("both", "M1", read_heads, 1, 1))
    rows = []
    for followup_index in range(count):
        followup, initial = _followup_case_and_environment(case, followup_index)
        followup["_condition"] = pair["condition"]
        actor_cfg = dict(cfg, condition=pair["condition"], evaluation_layer=layer)
        for arm, memory_label, heads, writer_flag, actor_flag in arms:
            result_memory = pair[memory_label]
            memory = copy.deepcopy(result_memory["memory"])
            before = stable_hash(memory)
            payload = {"case": followup, "condition": pair["condition"],
                       "memory": memory, "memory_valid": result_memory["valid"],
                       "memory_hash": before, "heads": [list(head) for head in heads],
                       "initial_env": initial, "tau_hash": pair["tau_hash"],
                       "cfg": actor_cfg, "version": "conflict_v3_memory_only_followup"}
            key = f'{case["source_id"]}-{pair["condition"]}-{layer}-{followup_index}-' + stable_hash(payload)[:20]
            if result_memory["valid"]:
                actor_result = checkpoint_item(
                    "memory_propagation", key, payload,
                    lambda: run_downstream_actor(
                        backend, copy.deepcopy(followup), copy.deepcopy(memory), heads=heads,
                        cfg=actor_cfg, initial_env=copy.deepcopy(initial)),
                )
                failure_reason = None
            else:
                # Invalid memory is a pipeline failure, not a model action or excluded row.
                actor_result, failure_reason = None, "invalid_writer_memory"
            if stable_hash(memory) != before:
                raise RuntimeError("Downstream evaluation modified its input memory")
            result = actor_result or {}
            success = bool(result.get("task_success", False))
            row = {
                "source_id": case["source_id"], "split": case["split"],
                "condition": pair["condition"], "followup_id": followup["followup_id"],
                "followup_index": followup_index, "arm": arm, "evaluation_layer": layer,
                "writer_intervention": writer_flag, "actor_intervention": actor_flag,
                "success": success, "task_success": success,
                "memory_valid": bool(result_memory["valid"]),
                "memory_value_correct": result_memory["metrics"]["target_correct"],
                "memory_source_correct": result_memory["metrics"]["target_source_correct"],
                "memory_joint_correct": result_memory["metrics"]["target_joint_correct"],
                "memory_conflict_error": result_memory["metrics"]["target_conflict_error"],
                "memory_hash": before, "environment_hash": stable_hash(initial),
                "tau_hash": pair["tau_hash"], "failure_reason": failure_reason,
                "writer_format_failure": not result_memory["valid"],
                "actor_format_failure": bool(result.get("format_failure", False)),
                "format_failure": bool(not result_memory["valid"] or result.get("format_failure", False)),
                "actor_result": actor_result,
                "input_memory": memory, "read_heads": [list(head) for head in heads],
                "memory_origin": "model_generated_from_controlled_context",
                "gold_profile_reinjected": False,
            }
            for metric in ("binding_correct", "binding_distractor", "order_attempted",
                           "distractor_action", "accepted_wrong_city", "accepted_conflict_city",
                           "prerequisite_path_valid", "strict_task_success", "tool_execution_valid"):
                row[metric] = bool(result.get(metric, False)) if layer == "end_to_end" or metric not in ("prerequisite_path_valid", "strict_task_success") else None
            row["decision_action_success"] = success if layer == "controlled_decision" else None
            row["subgoal_progress"] = result.get("subgoal_progress", 0.0) if layer == "end_to_end" else None
            rows.append(row)
    return rows


def evaluate_memory_pipeline(backend, cases, histories, read_heads, write_heads, cfg):
    """Always evaluate baseline generated-memory propagation in BOTH conditions.

    Returns JSON-compatible keys:
      propagation: all source x condition x followup x actual arm x layer rows;
      memory_pairs: condition -> source_id -> {M0, optional M1, provenance};
      writer_rows: underlying generated memories and separate Writer metrics;
      factorial_available: true only when both nonempty head sets are supplied;
      factorial / controlled_factorial: corresponding layer subsets, which may
        contain only baseline/partial arms; consult factorial_available.
    Observations remain clustered by source_id, not by followup or condition.
    """
    source_ids = [case["source_id"] for case in cases]
    if not source_ids or len(source_ids) != len(set(source_ids)):
        raise ValueError("Pipeline needs unique nonempty source cases")
    layers = tuple(cfg.get("pipeline_layers", ("end_to_end", "controlled_decision")))
    if not layers or len(layers) != len(set(layers)) or any(layer not in ("end_to_end", "controlled_decision") for layer in layers):
        raise ValueError("pipeline_layers must be unique supported layers")
    head_sets = {"none": []}
    if write_heads:
        head_sets["H_W"] = write_heads
    writer_rows = evaluate_writer(backend, cases, histories, head_sets, cfg, conditions=CONDITIONS)
    indexed = {(row["source_id"], row["condition"], row["intervention"]): row["writer_result"]
               for row in writer_rows}
    pairs = {condition: {} for condition in CONDITIONS}
    propagation = []
    jobs = [(case, condition) for case in cases for condition in CONDITIONS]
    for original_case, condition in _writer_progress(jobs, "Memory  Follow-up tasks ", cfg):
        case = copy.deepcopy(original_case)
        case["_condition"] = condition
        source_id = case["source_id"]
        pair = {"source_id": source_id, "split": case["split"], "condition": condition,
                "tau_hash": stable_hash(histories[condition][source_id]),
                "write_heads": [list(head) for head in write_heads],
                "M0": copy.deepcopy(indexed[source_id, condition, "none"])}
        if write_heads:
            pair["M1"] = copy.deepcopy(indexed[source_id, condition, "H_W"])
            if pair["M0"]["prompt_hash"] != pair["M1"]["prompt_hash"]:
                raise RuntimeError("Writer intervention received a different history prompt")
        pairs[condition][source_id] = pair
        for layer in layers:
            propagation.extend(evaluate_factorial(backend, case, pair, read_heads,
                                                  dict(cfg, evaluation_layer=layer)))
    arms = ["baseline"] + (["writer_only"] if write_heads else []) + (["actor_only"] if read_heads else []) + (["both"] if read_heads and write_heads else [])
    return {"propagation": propagation, "memory_pairs": pairs, "writer_rows": writer_rows,
            "factorial_available": bool(read_heads and write_heads), "arms": arms,
            "conditions": list(CONDITIONS), "evaluation_layers": list(layers),
            "n_sources": len(cases), "followup_tasks": cfg.get("followup_tasks", 3),
            "factorial": [row for row in propagation if row["evaluation_layer"] == "end_to_end"],
            "controlled_factorial": [row for row in propagation if row["evaluation_layer"] == "controlled_decision"],
            "interpretation": "controlled_context_to_generated_memory_to_action; not clean-input natural errors",
            "gold_profile_reinjected": False}


In [ ]:
# Read-side evaluation and source-clustered descriptive/paired uncertainty.
import collections
import math
import random
import numpy as np


def _scalar_float(value):
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "cpu"):
        value = value.cpu()
    if hasattr(value, "item"):
        value = value.item()
    result = float(value)
    if not math.isfinite(result):
        raise ValueError("Nonfinite measurement; preserve the failing case and diagnose it")
    return result


def _evaluate_read_fresh(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    """Execute every planned source × condition × intervention, with checkpoints.

    head_sets maps human-readable labels to (layer, head) pairs. Actual full
    behavior is never filtered by token eligibility or initial attack success.
    Fixed-prefix margin is a secondary, explicitly eligible-only diagnostic.
    cfg is expected to contain model/revision/dtype/decoding identity so the
    checkpoint payload changes when the run configuration changes.
    """
    if not hasattr(head_sets, "items") or not head_sets:
        raise ValueError("head_sets must be a nonempty mapping label -> head pairs")
    rows = []
    for case in progress_cases(cases, "Actor episodes", cfg):
        for condition in conditions:
            memory = memory_for_condition(case, condition)
            snapshot = actor_snapshot(case, condition)
            gold_id = backend.one_token_id(snapshot["prompt"], snapshot["gold"])
            distractor_id = backend.one_token_id(snapshot["prompt"], snapshot["distractor"])
            eligible = gold_id is not None and distractor_id is not None and gold_id != distractor_id
            for intervention, selected in head_sets.items():
                heads = [list(pair) for pair in selected]
                condition_cfg = dict(cfg, condition=condition)
                payload = {"case": case, "memory": memory, "heads": heads,
                           "cfg": condition_cfg, "intervention": intervention,
                           "snapshot": snapshot, "version": "conflict_read_v3"}

                def compute(case=case, memory=memory, heads=heads,
                            condition_cfg=condition_cfg, snapshot=snapshot,
                            intervention=intervention, eligible=eligible,
                            gold_id=gold_id, distractor_id=distractor_id):
                    result = run_actor(backend, case, memory, heads=heads, cfg=condition_cfg)
                    result.update({"intervention": intervention, "heads": heads,
                                   "single_token_eligible": eligible,
                                   "gold_token_id": gold_id, "distractor_token_id": distractor_id,
                                   "snapshot_margin": _scalar_float(backend.margin(snapshot, heads=heads)) if eligible and cfg.get("compute_snapshot_margin", True) else None,
                                   "snapshot_origin": snapshot["snapshot_origin"]})
                    return result

                key = f"{case['source_id']}-{condition}-{intervention}"
                rows.append(checkpoint_item("read_eval", key, payload, compute))
    return rows


def same_layer_random_heads(selected, n_layers, n_heads, seed):
    """Match each selected layer's head count, excluding ALL selected heads.

    Raise if the requested disjoint control cannot be formed. Never silently
    relax exclusion or layer matching. Repeat with distinct recorded seeds.
    """
    selected = [tuple(pair) for pair in selected]
    if len(set(selected)) != len(selected):
        raise ValueError("selected contains duplicate heads")
    if n_layers < 1 or n_heads < 1:
        raise ValueError("n_layers and n_heads must be positive")
    for layer, head in selected:
        if type(layer) is not int or type(head) is not int or not (0 <= layer < n_layers and 0 <= head < n_heads):
            raise ValueError(f"Invalid selected head {(layer, head)}")
    by_layer = collections.defaultdict(set)
    for layer, head in selected:
        by_layer[layer].add(head)
    rng = random.Random(int(seed))
    control = []
    for layer in sorted(by_layer):
        available = [head for head in range(n_heads) if head not in by_layer[layer]]
        count = len(by_layer[layer])
        if len(available) < count:
            raise ValueError(f"Layer {layer}: {count} selected heads but only {len(available)} nonselected heads; no disjoint layer-matched control")
        control.extend((layer, head) for head in rng.sample(available, count))
    return sorted(control)


def _evaluate_controlled_read_fresh(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    """All-source complete-action evaluation. Never uses the token probe prefix."""
    if not head_sets:
        raise ValueError("A nonempty head-set mapping is required")
    rows = []
    for case in progress_cases(cases, "Actor episodes", cfg):
        for condition in conditions:
            memory = memory_for_condition(case, condition)
            condition_cfg = dict(cfg, condition=condition, evaluation_layer="controlled_decision")
            for label, heads in head_sets.items():
                payload = {"case": case, "memory": memory, "heads": [list(h) for h in heads],
                           "cfg": condition_cfg, "intervention": label,
                           "version": "conflict_controlled_v3"}
                key = f"{case['source_id']}-{condition}-{label}"
                result = checkpoint_item("controlled_read", key, payload,
                    lambda: run_controlled_actor(backend, case, memory, heads=heads, cfg=condition_cfg))
                rows.append({**result, "intervention": label, "heads": [list(h) for h in heads]})
    return rows


def layered_read_effects(end_to_end, controlled, arm="H_R", n_boot=2000):
    """Do not pool controlled decisions and autonomous trajectories."""
    effects = []
    for layer, rows, metrics in (
        ("end_to_end", end_to_end, ("task_success", "strict_task_success", "binding_correct", "binding_distractor")),
        ("controlled_decision", controlled, ("decision_action_success", "binding_correct", "binding_distractor")),
    ):
        for condition in sorted({r["condition"] for r in rows}):
            for metric in metrics:
                effects.append(dict(evaluation_layer=layer, condition=condition,
                    **paired_effect(rows, "intervention", "none", arm, metric=metric,
                                    filters={"condition": condition}, n_boot=n_boot)))
    return effects


def actor_diagnostics(rows):
    """Descriptive process failures; never remove episodes or alter scoring."""
    diagnostics = []
    for row in rows:
        errors = [event["observation"].get("error") for event in row["events"]
                  if isinstance(event.get("observation"), dict)
                  and not event["observation"].get("ok", False)]
        reserved = sum(event.get("action", {}).get("tool") == "reserve_item"
                       and bool((event.get("observation") or {}).get("ok"))
                       for event in row["events"] if isinstance(event.get("action"), dict))
        diagnostics.append({
            key: row[key] for key in ("source_id", "condition", "intervention", "template_id",
                                     "preferred_available", "position_order", "task_success",
                                     "format_failure", "terminal", "action_count", "budget_exhausted")
        })
        diagnostics[-1].update(tool_error_count=len(errors),
                               out_of_stock_count=errors.count("out_of_stock"),
                               invalid_reservation_count=errors.count("invalid_reservation"),
                               successful_reservations=reserved,
                               accepted_order_count=row["accepted_order_count"],
                               gold_city=row["gold_city"], gold_item=row["gold_item"])
        diagnostics[-1].update({key: row.get(key) for key in (
            "binding_category", "binding_correct", "binding_distractor", "order_attempted",
            "subgoal_progress", "prerequisite_path_valid", "strict_task_success",
            "first_tool_error_step", "first_tool_error")})
    return diagnostics


def _passes_filters(row, filters):
    if filters is None:
        return True
    if callable(filters):
        return bool(filters(row))
    for key, expected in filters.items():
        actual = row.get(key)
        if isinstance(expected, (list, tuple, set, frozenset)):
            if actual not in expected:
                return False
        elif actual != expected:
            return False
    return True


def _get_metric(row, metric):
    if metric in row:
        return row[metric]
    return row.get("metrics", {}).get(metric)


def _source_bootstrap(values, seed, n_boot):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all():
        raise ValueError("Bootstrap requires a nonempty finite 1-D source-level array")
    if not isinstance(n_boot, int) or n_boot < 1:
        raise ValueError("n_boot must be a positive integer")
    rng = np.random.default_rng(int(seed))
    # Chunk so large source counts do not allocate n_boot × n_sources at once.
    estimates = []
    for start in range(0, n_boot, 256):
        count = min(256, n_boot - start)
        indices = rng.integers(0, len(values), size=(count, len(values)))
        estimates.extend(values[indices].mean(axis=1).tolist())
    lower, upper = np.quantile(np.asarray(estimates), [0.025, 0.975])
    return {"estimate": float(values.mean()), "ci_low": float(lower), "ci_high": float(upper),
            "n_sources": int(len(values)), "n_boot": n_boot,
            "ci_method": "percentile_source_bootstrap_95pct",
            "small_sample_warning": bool(len(values) < 20),
            "degenerate_interval": bool(lower == upper),
            "boundary_note": "A zero-width bootstrap interval does not imply zero population uncertainty" if lower == upper else None}


def paired_effect(rows, arm_col, arm0, arm1, metric="task_success", filters=None, seed=11, n_boot=2000):
    """Mean(arm1 − arm0), bootstrapping source cases after within-source means.

    Repeated future tasks, fields, or seeds never become independent units here.
    A missing arm/source/metric is an error, not silent complete-case filtering.
    Equal repeat counts are required within each paired source; if followup_id
    exists, its set must also match. Use filters to select the planned contrast.
    """
    if arm0 == arm1:
        raise ValueError("arm0 and arm1 must differ")
    grouped = {arm0: collections.defaultdict(list), arm1: collections.defaultdict(list)}
    for row in rows:
        arm = row.get(arm_col)
        if arm not in grouped or not _passes_filters(row, filters):
            continue
        if "source_id" not in row:
            raise ValueError("Each observation requires source_id")
        value = _get_metric(row, metric)
        if value is None:
            raise ValueError(f"Missing {metric} for {row['source_id']} / {arm}")
        grouped[arm][row["source_id"]].append((row, _scalar_float(value)))
    sources0, sources1 = set(grouped[arm0]), set(grouped[arm1])
    if not sources0 or sources0 != sources1:
        raise ValueError(f"Unpaired or empty sources: arm0={len(sources0)}, arm1={len(sources1)}, missing_in_arm1={sorted(sources0-sources1)[:5]}, missing_in_arm0={sorted(sources1-sources0)[:5]}")
    differences, means0, means1 = [], [], []
    count_rows = 0
    for source in sorted(sources0):
        entries0, entries1 = grouped[arm0][source], grouped[arm1][source]
        if len(entries0) != len(entries1):
            raise ValueError(f"Unmatched repeat counts for source {source}: {len(entries0)} vs {len(entries1)}")
        if any("followup_id" in row for row, _ in entries0 + entries1):
            ids0 = collections.Counter(row.get("followup_id") for row, _ in entries0)
            ids1 = collections.Counter(row.get("followup_id") for row, _ in entries1)
            if ids0 != ids1:
                raise ValueError(f"Unpaired followup_id values for source {source}")
        mean0 = sum(v for _, v in entries0) / len(entries0)
        mean1 = sum(v for _, v in entries1) / len(entries1)
        means0.append(mean0)
        means1.append(mean1)
        differences.append(mean1 - mean0)
        count_rows += len(entries0) + len(entries1)
    result = _source_bootstrap(differences, seed, n_boot)
    result.update({"arm_col": arm_col, "arm0": arm0, "arm1": arm1, "metric": metric,
                   "direction": "arm1_minus_arm0", "n_rows": count_rows,
                   "arm0_mean": float(np.mean(means0)), "arm1_mean": float(np.mean(means1)),
                   "n_unpaired_sources": 0, "seed": int(seed)})
    return result


def bootstrap_summary(rows, group_cols, metrics, n_boot=2000):
    """Descriptive group means with a source-clustered 95% percentile interval.

    Returns one row per group × metric. Missing optional metrics (e.g. ineligible
    snapshot margins) are counted explicitly. For confirmatory arm comparisons,
    use paired_effect; these separate intervals do not estimate a difference.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    if isinstance(metrics, str):
        metrics = [metrics]
    groups = collections.defaultdict(list)
    for row in rows:
        if "source_id" not in row:
            raise ValueError("Each observation requires source_id")
        key = tuple(row.get(col) for col in group_cols)
        groups[key].append(row)
    summaries = []
    for group_index, (key, members) in enumerate(sorted(groups.items(), key=lambda item: repr(item[0]))):
        for metric_index, metric in enumerate(metrics):
            per_source = collections.defaultdict(list)
            n_missing = 0
            for row in members:
                value = _get_metric(row, metric)
                if value is None:
                    n_missing += 1
                else:
                    per_source[row["source_id"]].append(_scalar_float(value))
            summary = dict(zip(group_cols, key))
            summary.update({"metric": metric, "n_rows": len(members),
                            "n_missing_rows": n_missing, "n_valid_rows": len(members) - n_missing,
                            "n_all_sources": len({row["source_id"] for row in members})})
            values = [sum(values) / len(values) for _, values in sorted(per_source.items())]
            if values:
                summary.update(_source_bootstrap(values, 11 + 1009 * group_index + metric_index, n_boot))
            else:
                summary.update({"estimate": None, "ci_low": None, "ci_high": None,
                                "n_sources": 0, "n_boot": n_boot, "ci_method": "no_valid_values"})
            summaries.append(summary)
    return summaries

# Standalone evaluators remain available before the explicit reuse wrappers load.


In [ ]:
import math

def _behavioral_number(row, metric):
    value = row.get(metric)
    if value is None or not isinstance(value, (bool, int, float)) or (not math.isfinite(float(value))):
        raise ValueError(f"Missing/nonfinite behavioral metric {metric}: {row.get('source_id')}")
    if metric.endswith('_count'):
        if value < 0 or int(value) != value:
            raise ValueError(f'Invalid behavioral count {metric}')
    elif value not in (0, 1):
        raise ValueError(f'Behavioral indicator {metric} must be 0 or 1')
    return float(value)

def _behavioral_index(rows, source_ids, conditions, intervention, metrics, layer=None):
    """Fail closed on missing, duplicate or unexpected observations; never filter failures."""
    expected = {(source, condition) for source in source_ids for condition in conditions}
    indexed = {}
    for row in rows:
        key = (row.get('source_id'), row.get('condition'))
        if key not in expected or row.get('intervention') != intervention:
            raise ValueError(f"Unexpected behavioral observation: {key}/{row.get('intervention')}")
        if key in indexed:
            raise ValueError(f'Duplicate behavioral observation: {key}')
        if layer is not None and row.get('evaluation_layer') != layer:
            raise ValueError(f'Wrong evaluation layer: {key}')
        for metric in metrics:
            _behavioral_number(row, metric)
        if 'binding_correct' in metrics and 'binding_distractor' in metrics:
            if row['binding_correct'] and row['binding_distractor']:
                raise ValueError(f'Contradictory binding indicators: {key}')
        indexed[key] = row
    if set(indexed) != expected:
        raise ValueError(f'Missing planned behavioral observations: {sorted(expected - set(indexed))}')
    return indexed

def _behavioral_means(indexed, condition, metrics):
    rows = [row for (_, cond), row in indexed.items() if cond == condition]
    if not rows:
        raise ValueError('Cannot summarize an empty behavioral condition')
    return {name: sum((_behavioral_number(row, metric) for row in rows)) / len(rows) for metric, name in metrics.items()}

def read_phenomenon_gate(rows, cfg):
    """Paired untreated pilot: clean correct actions must become genuine distractor bindings.

    A malformed output or missing order never counts as a model-generated conflict adoption.
    n_pilot (when supplied) also detects sources absent from both conditions.
    """
    conditions = ('aligned', 'conflict')
    baseline = [r for r in rows if r.get('intervention') == 'none' and r.get('condition') in conditions]
    source_ids = sorted({r.get('source_id') for r in baseline if isinstance(r.get('source_id'), str)})
    if not source_ids or ('n_pilot' in cfg and len(source_ids) != cfg['n_pilot']):
        raise ValueError('Pilot baseline does not contain every planned source')
    metrics = ('decision_action_success', 'binding_correct', 'binding_distractor', 'format_failure')
    indexed = _behavioral_index(baseline, source_ids, conditions, 'none', metrics, 'controlled_decision')
    clean = [indexed[source, 'aligned'] for source in source_ids]
    attacked = [indexed[source, 'conflict'] for source in source_ids]
    success = sum((r['decision_action_success'] for r in clean)) / len(clean)
    format_failure = sum((r['format_failure'] for r in clean)) / len(clean)
    errors = sum((bool(c['decision_action_success'] and c['binding_correct'] and (not c['format_failure']) and a['binding_distractor'] and (not a['format_failure'])) for c, a in zip(clean, attacked)))
    minimum = int(cfg.get('min_read_conflict_errors', 2))
    if minimum < 1:
        raise ValueError('min_read_conflict_errors must be at least one')
    min_success = cfg.get('min_controlled_success', 0.8)
    max_format_failure = cfg.get('max_pilot_format_failure', 0.2)
    reasons = []
    if success < min_success:
        reasons.append('clean_controlled_capability_below_threshold')
    if format_failure > max_format_failure:
        reasons.append('clean_format_failure_above_threshold')
    if errors < minimum:
        reasons.append('insufficient_paired_conflict_adoptions')
    return {'eligible': not reasons, 'reasons': reasons, 'n_sources': len(source_ids), 'paired_conflict_adoptions': errors, 'paired_binding_regressions': errors, 'min_errors': minimum, 'clean_success': success, 'format_failure': format_failure, 'min_success': min_success, 'max_format_failure': max_format_failure, 'conflict_binding_error': sum((r['binding_distractor'] and (not r['format_failure']) for r in attacked)) / len(attacked), 'evaluation_layer': 'controlled_decision', 'gate_version': 'paired_target_conflict_v3', 'interpretation': 'feasibility_gate_only_not_a_significance_test'}

def _behavioral_cases(cases, cfg):
    ids = [case.get('source_id') for case in cases]
    if not ids or any((not isinstance(source, str) or not source for source in ids)) or len(set(ids)) != len(ids):
        raise ValueError('Behavioral selection requires nonempty unique planned sources')
    if any((case.get('split') != 'dev' for case in cases)):
        raise ValueError('Behavioral selection accepts development cases only')
    if 'n_dev' in cfg and len(cases) != cfg['n_dev']:
        raise ValueError('Behavioral selection must use all planned development cases')
    return ids

def _behavioral_candidates(backend, ranking, metric_names):
    ranked = ranking['ranked_heads']
    n_layers, n_heads = (backend.model.cfg.n_layers, backend.model.cfg.n_heads)
    seen = set()
    for head in ranked:
        if len(head) != 2 or any((type(x) is not int for x in head)):
            raise ValueError('Ranking contains invalid head coordinates')
        layer, index = head
        if not (0 <= layer < n_layers and 0 <= index < n_heads) or tuple(head) in seen:
            raise ValueError('Ranking contains duplicate or out-of-bounds heads')
        seen.add(tuple(head))
    candidates = []
    for k in (1, 2, 4, 8, 16):
        heads = [list(h) for h in ranked[:k]]
        counts = {}
        for layer, _ in heads:
            counts[layer] = counts.get(layer, 0) + 1
        possible = len(heads) == k and all((count * 2 <= n_heads for count in counts.values()))
        reasons = []
        if len(heads) != k:
            reasons.append('ranking_shorter_than_k')
        elif not possible:
            reasons.append('disjoint_layer_matched_random_control_unavailable')
        candidates.append({'k': k, 'heads': heads, 'eligible': False, 'reasons': reasons, 'disjoint_random_control_possible': possible, 'evaluated_stages': [], **{name: None for name in metric_names}, **{name + '_change': None for name in metric_names}})
    return candidates

def _behavioral_record(candidate, prefix, means, baseline):
    for name, value in means.items():
        candidate[prefix + name] = value
        candidate[prefix + name + '_change'] = value - baseline[name]

def _behavioral_preserve(candidate, prefix, higher, lower):
    """Strict aggregate preservation over the same complete, paired source set."""
    for metric in higher:
        if candidate[prefix + metric + '_change'] < -1e-12:
            candidate['reasons'].append(prefix + metric + '_decreased')
    for metric in lower:
        if candidate[prefix + metric + '_change'] > 1e-12:
            candidate['reasons'].append(prefix + metric + '_increased')

def _behavioral_finish(name, ranking, candidates, baseline, gate, gain_key):
    eligible = [row for row in candidates if row['eligible']]
    selected = max(eligible, key=lambda row: (row[gain_key], -row['k'])) if eligible else None
    result = {'name': name, 'seed': ranking['seed'], 'candidates': candidates, 'selected': selected, 'heads': selected['heads'] if selected else [], 'eligible': selected is not None, 'baseline': baseline, 'gate': gate, 'n_sources': candidates[0]['n_sources'], 'selection_method': 'all_source_complete_behavior_v1', 'reasons': [] if selected else ['no_candidate_passed_behavioral_constraints'], 'primary_seed_policy': 'caller_keeps_predeclared_primary_seed_no_best_of_seeds'}
    atomic_json(RUN_DIR / 'search' / f"{name}-seed{ranking['seed']}-behavioral-dev.json", result)
    return result

def select_read_heads_behavioral(backend, ranking, cases, cfg, phenomenon_gate):
    """Evaluate fixed small K on all dev sources; token margins never select heads.

    Clean controlled damage short-circuits expensive downstream evaluations.
    A failed pilot phenomenon gate still permits diagnosis, never a selected arm.
    End-to-end baseline is computed only if some candidate passes clean control.
    """
    source_ids = _behavioral_cases(cases, cfg)
    conditions = ('aligned', 'conflict')
    controlled_metrics = {'decision_action_success': 'success', 'binding_correct': 'binding_correct', 'binding_distractor': 'binding_distractor', 'format_failure': 'format_failure'}
    end_metrics = {'task_success': 'success', 'strict_task_success': 'strict_success', 'format_failure': 'format_failure'}
    names = [f'{condition}_{stage}_{metric}' for condition in ('clean', 'conflict') for stage, metrics in (('controlled', controlled_metrics), ('end_to_end', end_metrics)) for metric in metrics.values()]
    candidates = _behavioral_candidates(backend, ranking, names)
    for row in candidates:
        row['n_sources'] = len(source_ids)
    baseline_rows = evaluate_controlled_read(backend, cases, {'none': []}, cfg, conditions=conditions)
    indexed = _behavioral_index(baseline_rows, source_ids, conditions, 'none', controlled_metrics, 'controlled_decision')
    baseline = {'clean_controlled': _behavioral_means(indexed, 'aligned', controlled_metrics), 'conflict_controlled': _behavioral_means(indexed, 'conflict', controlled_metrics), 'clean_end_to_end': None, 'conflict_end_to_end': None}
    for row in candidates:
        if row['reasons']:
            continue
        label = f"behavioral_read_seed{ranking['seed']}_k{row['k']}"
        raw = evaluate_controlled_read(backend, cases, {label: row['heads']}, cfg, conditions=('aligned',))
        measured = _behavioral_index(raw, source_ids, ('aligned',), label, controlled_metrics, 'controlled_decision')
        _behavioral_record(row, 'clean_controlled_', _behavioral_means(measured, 'aligned', controlled_metrics), baseline['clean_controlled'])
        row['evaluated_stages'].append('clean_controlled')
        _behavioral_preserve(row, 'clean_controlled_', ('success', 'binding_correct'), ('format_failure', 'binding_distractor'))
        if row['clean_controlled_success'] < cfg.get('min_controlled_success', 0.8):
            row['reasons'].append('clean_controlled_capability_below_threshold')
        if row['reasons']:
            row['reasons'].append('later_stages_skipped_after_clean_damage')
            if not phenomenon_gate.get('eligible', False):
                row['reasons'].append('read_phenomenon_gate_failed')
            continue
        if not phenomenon_gate.get('eligible', False):
            row['reasons'].extend(['read_phenomenon_gate_failed', 'not_evaluated_no_conflict_adoptions'])
            continue
        raw = evaluate_controlled_read(backend, cases, {label: row['heads']}, cfg, conditions=('conflict',))
        measured = _behavioral_index(raw, source_ids, ('conflict',), label, controlled_metrics, 'controlled_decision')
        _behavioral_record(row, 'conflict_controlled_', _behavioral_means(measured, 'conflict', controlled_metrics), baseline['conflict_controlled'])
        row['evaluated_stages'].append('conflict_controlled')
        _behavioral_preserve(row, 'conflict_controlled_', ('success',), ('format_failure', 'binding_distractor'))
        if row['conflict_controlled_binding_correct_change'] <= 1e-12:
            row['reasons'].append('no_distractor_binding_improvement')
        if baseline['clean_end_to_end'] is None:
            raw = evaluate_read(backend, cases, {'none': []}, cfg, conditions=conditions)
            measured = _behavioral_index(raw, source_ids, conditions, 'none', end_metrics, 'end_to_end')
            for condition, short in zip(conditions, ('clean', 'conflict')):
                baseline[short + '_end_to_end'] = _behavioral_means(measured, condition, end_metrics)
        raw = evaluate_read(backend, cases, {label: row['heads']}, cfg, conditions=conditions)
        measured = _behavioral_index(raw, source_ids, conditions, label, end_metrics, 'end_to_end')
        for condition, short in zip(conditions, ('clean', 'conflict')):
            prefix = short + '_end_to_end_'
            _behavioral_record(row, prefix, _behavioral_means(measured, condition, end_metrics), baseline[short + '_end_to_end'])
            _behavioral_preserve(row, prefix, ('success', 'strict_success'), ('format_failure',))
        row['evaluated_stages'].append('end_to_end')
        if not phenomenon_gate.get('eligible', False):
            row['reasons'].append('read_phenomenon_gate_failed')
        row['eligible'] = not row['reasons']
    return _behavioral_finish('read', ranking, candidates, baseline, phenomenon_gate, 'conflict_controlled_binding_correct_change')

def select_writer_heads_behavioral(backend, ranking, cases, histories, cfg, writer_gate):
    """Future Writer selection on frozen histories, with validity and semantic retention.

    Wrong values, omissions, duplicates and wrong sources cannot be traded for
    target correction. Failed outputs stay in every all-source denominator.
    """
    source_ids = _behavioral_cases(cases, cfg)
    conditions = ('aligned', 'conflict')
    metrics = {key: key for key in ('target_correct', 'target_joint_correct', 'format_failure', 'omission', 'duplicate_count', 'source_mismatch_count', 'unsupported_count')}
    names = [condition + '_' + metric for condition in ('clean', 'conflict') for metric in metrics]
    candidates = _behavioral_candidates(backend, ranking, names)
    for row in candidates:
        row['n_sources'] = len(source_ids)
    raw = evaluate_writer(backend, cases, histories, {'none': []}, cfg, conditions=conditions)
    indexed = _behavioral_index(raw, source_ids, conditions, 'none', metrics)
    baseline = {short: _behavioral_means(indexed, condition, metrics) for condition, short in zip(conditions, ('clean', 'conflict'))}
    lower = ('format_failure', 'omission', 'duplicate_count', 'source_mismatch_count', 'unsupported_count')
    for row in candidates:
        if row['reasons']:
            continue
        label = f"behavioral_write_seed{ranking['seed']}_k{row['k']}"
        raw = evaluate_writer(backend, cases, histories, {label: row['heads']}, cfg, conditions=('aligned',))
        measured = _behavioral_index(raw, source_ids, ('aligned',), label, metrics)
        _behavioral_record(row, 'clean_', _behavioral_means(measured, 'aligned', metrics), baseline['clean'])
        row['evaluated_stages'].append('clean_writer')
        _behavioral_preserve(row, 'clean_', ('target_correct', 'target_joint_correct'), lower)
        if 1 - row['clean_format_failure'] < cfg.get('min_writer_valid', 0.8):
            row['reasons'].append('clean_writer_validity_below_threshold')
        if row['clean_target_joint_correct'] < cfg.get('min_writer_clean_correct', 0.8):
            row['reasons'].append('clean_writer_target_retention_below_threshold')
        if row['reasons']:
            row['reasons'].append('conflict_writer_skipped_after_clean_damage')
            if not writer_gate.get('eligible', False):
                row['reasons'].append('writer_gate_failed')
            continue
        raw = evaluate_writer(backend, cases, histories, {label: row['heads']}, cfg, conditions=('conflict',))
        measured = _behavioral_index(raw, source_ids, ('conflict',), label, metrics)
        _behavioral_record(row, 'conflict_', _behavioral_means(measured, 'conflict', metrics), baseline['conflict'])
        row['evaluated_stages'].append('conflict_writer')
        _behavioral_preserve(row, 'conflict_', ('target_correct',), lower)
        if row['conflict_target_joint_correct_change'] <= 1e-12:
            row['reasons'].append('no_distractor_target_correction_improvement')
        if not writer_gate.get('eligible', False):
            row['reasons'].append('writer_gate_failed')
        row['eligible'] = not row['reasons']
    return _behavioral_finish('write', ranking, candidates, baseline, writer_gate, 'conflict_target_joint_correct_change')


In [ ]:
"""Conflict-v3 evaluation, paired gates, exports, and immutable test protocol."""
import inspect
import csv
import copy


def progress_cases(cases, description, cfg):
    bar = globals().get("tqdm")
    return bar(cases, desc=description, disable=not cfg.get("show_progress", True)) if bar else cases


def evaluate_read(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    return _evaluate_read_fresh(backend, cases, head_sets, cfg, conditions)


def evaluate_controlled_read(backend, cases, head_sets, cfg, conditions=CONDITIONS):
    return _evaluate_controlled_read_fresh(backend, cases, head_sets, cfg, conditions)


def histories_from_actor_rows(backend, cases, rows, cfg):
    return collect_fixed_histories(backend, cases, CONDITIONS, cfg, actor_rows=rows)


def capability_gate(rows, cfg, controlled=False):
    clean = [r for r in rows if r["condition"] == "aligned" and r["intervention"] == "none"]
    if len(clean) != cfg["n_pilot"] or len({r["source_id"] for r in clean}) != len(clean):
        raise ValueError("Pilot capability needs every unique planned source")
    metric = "decision_action_success" if controlled else "task_success"
    success = sum(float(r[metric]) for r in clean) / len(clean)
    failure = sum(float(r["format_failure"]) for r in clean) / len(clean)
    threshold = cfg["min_controlled_success"] if controlled else cfg["min_pilot_success"]
    return {"eligible": success >= threshold and failure <= cfg["max_pilot_format_failure"],
            "n_sources": len(clean), "clean_success": success, "format_failure": failure,
            "min_success": threshold, "metric": metric,
            "interpretation": "project_feasibility_rule_not_statistical_significance"}


def writer_conflict_gate(rows, cfg):
    base = [r for r in rows if r["intervention"] == "none"]
    ids = sorted({r["source_id"] for r in base})
    if len(ids) != cfg["n_pilot"]:
        raise ValueError("Writer gate requires all pilot sources")
    metrics = ("target_correct", "target_joint_correct", "target_conflict_error", "format_failure")
    indexed = _behavioral_index(base, ids, CONDITIONS, "none", metrics)
    clean = [indexed[s, "aligned"] for s in ids]
    attack = [indexed[s, "conflict"] for s in ids]
    errors = sum(bool(c["target_joint_correct"] and not c["format_failure"]
                      and a["target_conflict_error"] and not a["format_failure"])
                 for c, a in zip(clean, attack))
    joint = sum(r["target_joint_correct"] for r in clean) / len(clean)
    valid = sum(not r["format_failure"] for r in clean) / len(clean)
    reasons = []
    if joint < cfg["min_writer_clean_correct"]:
        reasons.append("aligned_value_and_provenance_capability_below_threshold")
    if valid < cfg["min_writer_valid"]:
        reasons.append("aligned_writer_format_capability_below_threshold")
    if errors < cfg["min_writer_conflict_errors"]:
        reasons.append("insufficient_paired_conflict_adoptions")
    return {"eligible": not reasons, "reasons": reasons, "n_sources": len(ids),
            "paired_conflict_adoptions": errors, "clean_joint_correct": joint,
            "clean_valid_fraction": valid, "min_errors": cfg["min_writer_conflict_errors"],
            "error_origin": "model_generation_under_constructed_conflicting_note",
            "interpretation": "feasibility_rule_not_a_significance_test"}


def _atomic_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = sorted({k for row in rows for k in row})
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: canonical_json(v) if isinstance(v, (dict, list, tuple)) else v
                             for k, v in row.items()})
    os.replace(temporary, path)


def export_rows(name, rows, group_cols, metrics):
    atomic_json(RUN_DIR / "results" / (name + "-raw.json"), rows)
    summary = bootstrap_summary(rows, group_cols, metrics, CFG["bootstrap_reps"])
    # For one Bernoulli measurement per source, avoid misleading [0,0]/[1,1]
    # percentile intervals. Repeated followups retain source-cluster bootstrap.
    binary = {"task_success", "strict_task_success", "decision_action_success", "binding_correct",
              "binding_distractor", "binding_conflict", "accepted_conflict_city", "accepted_wrong_city",
              "target_correct", "target_joint_correct", "target_source_correct", "target_conflict_error",
              "target_distractor_error", "target_omission", "omission", "format_failure",
              "order_attempted", "tool_execution_valid", "prerequisite_path_valid"}
    for row in summary:
        n = row.get("n_sources", 0)
        if row["metric"] in binary and n and row["n_valid_rows"] == n:
            p, z = row["estimate"], 1.959963984540054
            denominator = 1 + z * z / n
            center = (p + z * z / (2 * n)) / denominator
            radius = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
            row.update(ci_low=max(0.0, center - radius), ci_high=min(1.0, center + radius),
                       ci_method="wilson_95pct_one_binary_observation_per_source", n_boot=0,
                       degenerate_interval=False, boundary_note=None)
    _atomic_csv(RUN_DIR / "results" / (name + "-summary.csv"), summary)
    atomic_json(RUN_DIR / "results" / (name + "-summary.json"), summary)
    return summary


def export_layered_read(name, end_to_end, controlled):
    summary = []
    for layer, rows, metrics in (
        ("end_to_end", end_to_end, ["task_success", "strict_task_success", "binding_correct",
            "binding_distractor", "accepted_conflict_city", "accepted_wrong_city", "format_failure",
            "order_attempted", "prerequisite_path_valid", "subgoal_progress", "snapshot_margin"]),
        ("controlled_decision", controlled, ["decision_action_success", "binding_correct",
            "binding_distractor", "accepted_conflict_city", "format_failure", "order_attempted", "tool_execution_valid"]),
    ):
        summary.extend(export_rows(name + "-" + layer, rows,
            ["evaluation_layer", "condition", "intervention"], metrics))
        _atomic_csv(RUN_DIR / "results" / (name + "-" + layer + "-strata.csv"),
            bootstrap_summary(rows, ["condition", "intervention", "template_id",
                "preferred_available", "position_order"], metrics, CFG["bootstrap_reps"]))
    return summary


def context_effects(rows, metrics, filters=None):
    return [paired_effect(rows, "condition", "aligned", "conflict", metric=metric,
                          filters=filters, n_boot=CFG["bootstrap_reps"]) for metric in metrics]


def intervention_effects(rows, metrics, arm_col="intervention", baseline="none"):
    results = []
    for condition in CONDITIONS:
        for arm in sorted({r[arm_col] for r in rows} - {baseline}):
            for metric in metrics:
                results.append(dict(condition=condition, **paired_effect(rows, arm_col, baseline, arm,
                    metric=metric, filters={"condition": condition}, n_boot=CFG["bootstrap_reps"])))
    return results


def export_pipeline(name, pipeline):
    rows = pipeline["propagation"]
    sources = set(pipeline["memory_pairs"][CONDITIONS[0]])
    if len(sources) != pipeline["n_sources"] or any(
            set(pipeline["memory_pairs"][c]) != sources for c in CONDITIONS):
        raise ValueError("Pipeline memory/source inventory is incomplete")
    planned = {(s, c, a, layer, i) for s in sources for c in CONDITIONS
               for a in pipeline["arms"] for layer in pipeline["evaluation_layers"]
               for i in range(pipeline["followup_tasks"])}
    observed = []
    for row in rows:
        key = (row["source_id"], row["condition"], row["arm"], row["evaluation_layer"], row["followup_index"])
        if row["followup_id"] != f'{row["source_id"]}:followup:{row["followup_index"]}':
            raise ValueError("Pipeline followup ID and index disagree")
        observed.append(key)
    if len(observed) != len(set(observed)) or set(observed) != planned:
        raise ValueError("Pipeline contains missing, duplicate or unexpected planned observations")
    atomic_json(RUN_DIR / "results" / (name + "-memories.json"), pipeline["memory_pairs"])
    metrics = ["task_success", "binding_correct", "binding_distractor", "accepted_conflict_city", "format_failure"]
    summary = export_rows(name, rows, ["evaluation_layer", "condition", "arm"], metrics)
    effects = []
    for layer in sorted({r["evaluation_layer"] for r in rows}):
        layered = [r for r in rows if r["evaluation_layer"] == layer]
        for e in context_effects(layered, metrics, {"arm": "baseline"}):
            effects.append(dict(evaluation_layer=layer, contrast="conflict_minus_aligned_baseline_memory", **e))
        effects.extend(dict(evaluation_layer=layer, **e) for e in
                       intervention_effects(layered, metrics, "arm", "baseline"))
        if pipeline["factorial_available"]:
            for condition in CONDITIONS:
                chosen = [r for r in layered if r["condition"] == condition]
                per_source = collections.defaultdict(lambda: collections.defaultdict(list))
                for r in chosen:
                    per_source[r["source_id"]][r["arm"]].append(r["task_success"])
                differences = []
                for source, arms in sorted(per_source.items()):
                    if set(arms) != {"baseline", "writer_only", "actor_only", "both"}:
                        raise ValueError("Incomplete factorial source: " + source)
                    m = {k: float(np.mean(v)) for k, v in arms.items()}
                    differences.append(m["both"] - m["writer_only"] - m["actor_only"] + m["baseline"])
                effects.append(dict(evaluation_layer=layer, condition=condition,
                    contrast="writer_actor_interaction", **_source_bootstrap(differences, 11, CFG["bootstrap_reps"])))
    atomic_json(RUN_DIR / "results" / (name + "-effects.json"), effects)
    _atomic_csv(RUN_DIR / "results" / (name + "-effects.csv"), effects)
    return summary


def _implementation_source(obj):
    if inspect.isclass(obj):
        return {"name": obj.__name__, "doc": obj.__doc__,
                "bases": [base.__qualname__ for base in obj.__bases__],
                "methods": {name: inspect.getsource(method) for name, method in vars(obj).items()
                            if inspect.isfunction(method)}}
    return inspect.getsource(obj)


def live_implementation_hash():
    return stable_hash({"definitions": {name: _implementation_source(globals()[name]) for name in IMPLEMENTATION_NAMES},
                        "task_templates": TASK_TEMPLATES, "conditions": CONDITIONS})


def verify_live_run():
    current = live_implementation_hash()
    fingerprint = stable_hash({"config": CFG, "model": MODEL_IDENTITY, "implementation": current})
    if current != IMPLEMENTATION_HASH or fingerprint != RUN_FINGERPRINT:
        raise RuntimeError("Live code/model/config changed; start a new named run instead of reusing checkpoints")


def freeze_protocol(read_selection, write_selection, read_controls, write_controls, data_hashes, gates):
    verify_live_run()
    if not globals().get("DEVELOPMENT_COMPLETE", False):
        raise RuntimeError("Finish the development baseline, selection and propagation cells first")
    protocol = {"schema": "agent_conflicting_context_v3", "run_fingerprint": RUN_FINGERPRINT,
        "implementation_hash": IMPLEMENTATION_HASH, "model": MODEL_IDENTITY, "config": CFG,
        "data_hashes": data_hashes, "conditions": CONDITIONS,
        "read_heads": read_selection["heads"] if read_selection and read_selection["eligible"] else [],
        "write_heads": write_selection["heads"] if write_selection and write_selection["eligible"] else [],
        "read_controls": read_controls, "write_controls": write_controls, "gates": gates,
        "primary_seed": CFG["search_seeds"][0],
        "primary_endpoints": {
            "actor": "controlled_decision.binding_distractor, conflict minus aligned, all sources",
            "writer": "target_joint_correct, conflict minus aligned, all sources",
            "propagation": "end_to_end.task_success, conflict minus aligned baseline memory, all source clusters"},
        "interventions": "if selected: fixed primary-seed heads versus none and layer-matched random controls",
        "uncertainty_unit": "source_episode; conditions, seeds and followups are not independent sources",
        "error_origin": "controlled erroneous retrieved note, with model-generated downstream memory",
        "authorized_update_limit": "source semantics unit-tested; authorized-update GPU preservation not tested in this two-condition study",
        "test_rule": "freeze before first test inference; resume same protocol only; no test-driven tuning"}
    protocol = json.loads(canonical_json(protocol))
    path = RUN_DIR / "frozen_protocol.json"
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != protocol:
        raise RuntimeError("Frozen protocol mismatch; do not overwrite or retune an existing test")
    atomic_json(path, protocol)
    return protocol


def verify_protocol(protocol, data):
    verify_live_run()
    if protocol["run_fingerprint"] != RUN_FINGERPRINT or protocol["implementation_hash"] != IMPLEMENTATION_HASH:
        raise RuntimeError("Test protocol identity mismatch")
    if protocol["data_hashes"] != {split: stable_hash(cases) for split, cases in data.items()}:
        raise RuntimeError("Data changed after protocol freeze")


In [ ]:
"""Complete two-condition experiment stages; all expensive stages are explicit."""


def cpu_semantic_smoke():
    cases = build_cases("pilot", 8, 1729, ["Paris", "Berlin", "London", "Boston"])
    for case in cases:
        aligned = memory_for_condition(case, "aligned")
        conflict = memory_for_condition(case, "conflict")
        changed = [(i, key) for i, (a, b) in enumerate(zip(aligned, conflict))
                   for key in a if a[key] != b[key]]
        assert len(changed) == 1 and changed[0][1] == "value"
        assert conflict[changed[0][0]]["kind"] == "retrieved_note"
        ref = authoritative_preferences(conflict)
        key = (case["target_name"], "delivery_city")
        assert ref[key]["value"] == case["target_city"]
        history = frozen_history(case, "conflict")
        right = [copy.deepcopy(ref[key])]
        score = score_memory(dict(case, _condition="conflict"), history, right, True)
        assert score["target_joint_correct"] and not score["target_conflict_error"]
        wrong = copy.deepcopy(right)
        wrong[0]["value"] = case["distractor_city"]
        score = score_memory(dict(case, _condition="conflict"), history, wrong, True)
        assert score["target_conflict_error"] and not score["target_correct"]
        wrong_source = copy.deepcopy(right)
        wrong_source[0]["source_event"] = case["source_id"] + ":note:0"
        score = score_memory(case, history, wrong_source, True)
        assert score["target_correct"] and not score["target_joint_correct"]
        updated = copy.deepcopy(history)
        update = dict(next(r for r in updated if r["kind"] == "verified_profile"),
                      value=case["updated_city"], source_event=case["source_id"] + ":profile:update",
                      sequence=20)
        note = dict(next(r for r in updated if r["kind"] == "retrieved_note"), sequence=21)
        updated += [update, note]
        assert authoritative_preferences(updated)[key]["value"] == case["updated_city"]
        prepared = prepare_legal_decision_state(case)
        env = OrderEnvironment(prepared["state"])
        response = env.step({"tool": "create_order", "customer": case["target_name"],
                            "item": prepared["item"], "city": case["distractor_city"],
                            "reservation": prepared["reservation"]})
        assert response["ok"] and response["order"]["city"] == case["distractor_city"]
    assert authoritative_preferences([{"kind": "retrieved_note", "value": "Paris"}]) == {}
    _, error = parse_json_output('{"tool":"check_stock","tool":"create_order"}', dict)
    assert error
    return {"passed": True, "source_cases": len(cases), "model_called": False,
            "checks": ["paired_only_note_value_changes", "source_authority", "latest_authorized_update",
                       "writer_value_source_separation", "wrong_city_not_repaired_by_tool", "strict_json"]}


def writer_metrics():
    return ["target_correct", "target_source_correct", "target_joint_correct", "target_conflict_error",
            "omission", "target_omission", "missing_count", "unsupported_count", "duplicate_count",
            "source_mismatch_count", "format_failure", "whole_memory_correct"]


def run_baseline_split(backend, cases, cfg, name):
    verify_live_run()
    if stable_hash(cfg) != stable_hash(CFG):
        raise RuntimeError("Stage configuration differs from the run manifest")
    print(name + ":  Two conditions.  Actor  Auto-execut and Control Full Actions ")
    end = evaluate_read(backend, cases, {"none": []}, cfg)
    controlled = evaluate_controlled_read(backend, cases, {"none": []}, cfg)
    actor_summary = export_layered_read(name + "-read", end, controlled)
    _atomic_csv(RUN_DIR / "results" / (name + "-actor-diagnostics.csv"), actor_diagnostics(end))
    histories = histories_from_actor_rows(backend, cases, end, cfg)
    atomic_json(RUN_DIR / "results" / (name + "-frozen-histories.json"), histories)
    print(name + ": Writer  Read the same real frozen experience ")
    writer = evaluate_writer(backend, cases, histories, {"none": []}, cfg)
    summary = export_rows(name + "-writer", writer, ["condition", "intervention"], writer_metrics())
    effects = []
    for layer, rows, metrics in (
        ("end_to_end", end, ["task_success", "strict_task_success", "binding_distractor", "accepted_conflict_city"]),
        ("controlled_decision", controlled, ["decision_action_success", "binding_distractor", "accepted_conflict_city"]),
        ("writer", writer, ["target_joint_correct", "target_correct", "target_conflict_error", "omission"]),
    ):
        effects.extend(dict(evaluation_layer=layer, **e) for e in context_effects(rows, metrics))
    atomic_json(RUN_DIR / "results" / (name + "-context-effects.json"), effects)
    _atomic_csv(RUN_DIR / "results" / (name + "-context-effects.csv"), effects)
    return {"end_to_end": end, "controlled": controlled, "histories": histories,
            "writer": writer, "actor_summary": actor_summary, "writer_summary": summary, "effects": effects}


def search_role(backend, role, train_cases, dev_cases, dev_histories, cfg, gate):
    verify_live_run()
    if role not in ("read", "write"):
        raise ValueError("Unknown role")
    if not cfg["run_head_search"] or not gate["eligible"]:
        print(role, " Search Skip: ", " Configure Close " if not cfg["run_head_search"] else gate.get("reasons", []))
        return {"rankings": [], "selections": [], "primary": None, "heads": [], "controls": {},
                "skip_reason": "disabled" if not cfg["run_head_search"] else "pilot_gate_failed"}
    histories = None
    if role == "write":
        train_rows = evaluate_read(backend, train_cases, {"none": []}, cfg)
        histories = histories_from_actor_rows(backend, train_cases, train_rows, cfg)
        atomic_json(RUN_DIR / "results" / "train-frozen-histories.json", histories)
    attacks, clean, exclusions = snapshot_pairs(backend, train_cases, "actor" if role == "read" else "writer", histories)
    atomic_json(RUN_DIR / "search" / (role + "-training-token-audit.json"),
                {"n_sources": len(train_cases), "n_valid_pairs": len(attacks), "exclusions": exclusions})
    if not attacks:
        print(role, " Theres no proper list. token Diagnosis pairing; behaviour baseline Remains as they are. ")
        return {"rankings": [], "selections": [], "primary": None, "heads": [], "controls": {},
                "skip_reason": "no_single_token_training_pairs"}
    rankings, selections = [], []
    for seed in cfg["search_seeds"]:
        ranking = train_mask(backend, attacks, clean, cfg, role, seed)
        selection = (select_read_heads_behavioral(backend, ranking, dev_cases, cfg, gate) if role == "read"
                     else select_writer_heads_behavioral(backend, ranking, dev_cases, dev_histories, cfg, gate))
        rankings.append(ranking)
        selections.append(selection)
        print(role, "seed", seed, "selected K:", len(selection["heads"]), "eligible:", selection["eligible"])
    primary = selections[0]  # fixed seed=11, never best-of-seeds
    heads = primary["heads"]
    controls = {"random_" + str(seed): same_layer_random_heads(heads, backend.model.cfg.n_layers,
                 backend.model.cfg.n_heads, seed) for seed in cfg["random_seeds"]} if heads else {}
    result = {"rankings": rankings, "selections": selections, "primary": primary,
              "heads": heads, "controls": controls, "skip_reason": None}
    atomic_json(RUN_DIR / "search" / (role + "-selection-status.json"),
                {k: v for k, v in result.items() if k != "rankings"})
    return result


def evaluate_selected(backend, cases, baseline, read_search, write_search, cfg, name):
    read_sets = {"none": []}
    if read_search["heads"]:
        read_sets.update({"H_R": read_search["heads"], **read_search["controls"]})
    end = evaluate_read(backend, cases, read_sets, cfg)
    controlled = evaluate_controlled_read(backend, cases, read_sets, cfg)
    actor_summary = export_layered_read(name + "-selected-read", end, controlled)
    for layer, rows, metrics in (("end_to_end", end, ["task_success", "strict_task_success", "binding_distractor"]),
                                ("controlled_decision", controlled, ["decision_action_success", "binding_distractor"])):
        atomic_json(RUN_DIR / "results" / (name + "-" + layer + "-intervention-effects.json"),
                    intervention_effects(rows, metrics))
    write_sets = {"none": []}
    if write_search["heads"]:
        write_sets.update({"H_W": write_search["heads"], **write_search["controls"]})
    if read_search["heads"]:
        write_sets["H_R_cross_role"] = read_search["heads"]
    writer = evaluate_writer(backend, cases, baseline["histories"], write_sets, cfg)
    writer_summary = export_rows(name + "-selected-writer", writer, ["condition", "intervention"], writer_metrics())
    atomic_json(RUN_DIR / "results" / (name + "-writer-intervention-effects.json"),
                intervention_effects(writer, ["target_joint_correct", "target_correct", "target_conflict_error", "omission"]))
    if write_search["heads"]:
        cross = {"none": [], "H_W_cross_role": write_search["heads"]}
        cross_end = evaluate_read(backend, cases, cross, cfg)
        cross_controlled = evaluate_controlled_read(backend, cases, cross, cfg)
        export_layered_read(name + "-writer-heads-on-actor", cross_end, cross_controlled)
    return {"end_to_end": end, "controlled": controlled, "writer": writer,
            "actor_summary": actor_summary, "writer_summary": writer_summary}


def show_summary(rows):
    # Never hide the middle conditions/metrics in a truncated pandas display.
    table = pd.DataFrame(rows)
    if not table.empty:
        wanted = [c for c in ("evaluation_layer", "condition", "intervention", "arm", "metric",
                   "estimate", "ci_low", "ci_high", "n_sources", "n_missing_rows", "ci_method") if c in table]
        with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 80):
            display(table[wanted])


def save_run_status(stage):
    status = {"schema": "conflict_v3", "stage": stage, "conditions": list(CONDITIONS),
              "run_fingerprint": RUN_FINGERPRINT, "model": MODEL_IDENTITY,
              "source_counts": {k: len(v) for k, v in globals().get("DATA", {}).items()},
              "gates": globals().get("GATES"),
              "read_heads": globals().get("READ_SEARCH", {}).get("heads", []),
              "write_heads": globals().get("WRITE_SEARCH", {}).get("heads", []),
              "dev_propagation_rows": len((globals().get("DEV_PIPELINE") or {}).get("propagation", [])),
              "factorial_available": bool((globals().get("DEV_PIPELINE") or {}).get("factorial_available", False)),
              "development_complete": globals().get("DEVELOPMENT_COMPLETE", False),
              "test_executed": (RUN_DIR / "test_complete.json").exists(),
              "previous_outputs_imported": False, "output_dir": str(RUN_DIR),
              "scope": "two matched constructed context conditions; no four-context concatenation"}
    atomic_json(RUN_DIR / "run_status.json", status)
    return status


def run_frozen_test(backend, data, read_search, write_search, gates, cfg):
    for role, search in (("read", read_search), ("write", write_search)):
        primary = search.get("primary")
        expected = primary["heads"] if primary and primary.get("eligible") else []
        if stable_hash(search["heads"]) != stable_hash(expected):
            raise RuntimeError(role + " heads differ from the primary development selection")
        if not expected and search["controls"]:
            raise RuntimeError(role + " has random controls without an actual selected intervention")
    protocol = freeze_protocol(read_search["primary"], write_search["primary"],
                               read_search["controls"], write_search["controls"], DATA_HASHES, gates)
    verify_protocol(protocol, data)
    if stable_hash(cfg) != stable_hash(protocol["config"]):
        raise RuntimeError("Test configuration differs from the frozen protocol")
    marker = {"protocol_hash": stable_hash(protocol), "test_sources": len(data["test"])}
    path = RUN_DIR / "test_started.json"
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != marker:
        raise RuntimeError("Existing test marker belongs to another protocol")
    atomic_json(path, marker)
    baseline = run_baseline_split(backend, data["test"], cfg, "test")
    frozen_read = {"heads": protocol["read_heads"], "controls": protocol["read_controls"]}
    frozen_write = {"heads": protocol["write_heads"], "controls": protocol["write_controls"]}
    selected = evaluate_selected(backend, data["test"], baseline, frozen_read, frozen_write, cfg, "test")
    pipeline = None
    if cfg["run_memory_pipeline"]:
        pipeline = evaluate_memory_pipeline(backend, data["test"], baseline["histories"],
                                            protocol["read_heads"], protocol["write_heads"], cfg)
        export_pipeline("test-memory-propagation", pipeline)
    atomic_json(RUN_DIR / "test_complete.json", {**marker, "completed": True,
                                               "baseline_only": not protocol["read_heads"] and not protocol["write_heads"]})
    return {"baseline": baseline, "selected": selected, "pipeline": pipeline, "protocol": protocol}


In [ ]:
def runtime_smoke(backend):
    """Small actual-model interface checks; no scientific outcome is assumed."""
    model = backend.model
    probe = backend.tokens('A short interface check: the city is')
    head = [(min(1, model.cfg.n_layers - 1), 0)]
    original_kind = backend.hook_kind
    try:
        with torch.no_grad():
            model.set_use_attn_result(False)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="z")):
                z_logits = model(probe, return_type="logits")[0, -1].float()
            model.set_use_attn_result(True)
            with model.hooks(fwd_hooks=ablation_hooks(model, head, hook_kind="result")):
                result_logits = model(probe, return_type="logits")[0, -1].float()
        max_difference = float((z_logits - result_logits).abs().max())
        if not torch.allclose(z_logits, result_logits, atol=0.25, rtol=0.02):
            raise RuntimeError(f"z/result ablation disagreement (max={max_difference}); inspect precision before search")
        model.set_use_attn_result(original_kind == "result")
        with torch.enable_grad():
            mask_logits = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=backend.device, requires_grad=True)
            drop = torch.sigmoid(mask_logits)
            with model.hooks(fwd_hooks=ablation_hooks(model, drop_mask=drop, hook_kind=original_kind)):
                score = model(probe, return_type="logits")[0, -1, :16].float().sum()
            score.backward()
            gradient_norm = float(mask_logits.grad.norm())
            if not math.isfinite(gradient_norm) or gradient_norm == 0:
                raise RuntimeError("Mask gradient is zero or nonfinite")
        # Different cache paths may expose tiny dtype differences. A mismatch
        # is a failed software check, never evidence of an intervention effect.
        generation_prompt = 'Complete one JSON object.\n{"check":'
        cache_agreement = None
        if backend.cfg.get("use_cache", True):
            cached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=True)
            uncached = backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
            cache_agreement = cached["text"] == uncached["text"]
            if not cache_agreement:
                raise RuntimeError("Cached/uncached greedy text differs. Inspect precision or start an explicitly uncached run (use_cache=False); do not silently continue.")
        else:
            backend.generate(generation_prompt, heads=head, max_new_tokens=8, use_cache=False)
        return {"passed": True, "z_result_max_abs_difference": max_difference,
                "mask_gradient_norm": gradient_norm, "cache_text_agreement": cache_agreement,
                "cache_test_skipped": not backend.cfg.get("use_cache", True),
                "cuda_peak_gb": torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None}
    finally:
        backend.hook_kind = original_kind
        model.set_use_attn_result(original_kind == "result")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
IMPLEMENTATION_NAMES = ['canonical_json', 'stable_hash', 'atomic_json', 'checkpoint_item', 'seed_everything', 'normalize_heads', 'ablation_hooks', 'TLBackend', 'eligible_snapshots', 'snapshot_pairs', 'gumbel_drop', '_save_search', 'train_mask', 'build_cases', '_preference_record', 'expected_city', 'distractor_city', 'memory_for_condition', 'authoritative_preferences', '_json_no_duplicates', 'parse_json_output', 'OrderEnvironment', '_task_goal', '_actor_protocol_examples', 'actor_prompt', '_accepted_city_metrics', 'run_actor', 'frozen_history', 'prepare_legal_decision_state', 'actor_snapshot', 'first_order_binding', 'score_actor_layers', 'run_controlled_actor', 'run_downstream_actor', '_latest_preference_records', '_memory_schema_error', '_writer_presented_history', '_writer_protocol_examples', 'writer_prompt', 'score_memory', 'run_writer', 'writer_snapshot', '_writer_progress', '_cached_writer_result', 'collect_fixed_histories', 'evaluate_writer', 'build_memory_pair', '_followup_case_and_environment', 'evaluate_factorial', 'evaluate_memory_pipeline', '_scalar_float', '_evaluate_read_fresh', 'same_layer_random_heads', '_evaluate_controlled_read_fresh', 'layered_read_effects', 'actor_diagnostics', '_passes_filters', '_get_metric', '_source_bootstrap', 'paired_effect', 'bootstrap_summary', '_behavioral_number', '_behavioral_index', '_behavioral_means', 'read_phenomenon_gate', '_behavioral_cases', '_behavioral_candidates', '_behavioral_record', '_behavioral_preserve', '_behavioral_finish', 'select_read_heads_behavioral', 'select_writer_heads_behavioral', 'progress_cases', 'evaluate_read', 'evaluate_controlled_read', 'histories_from_actor_rows', 'capability_gate', 'writer_conflict_gate', '_atomic_csv', 'export_rows', 'export_layered_read', 'context_effects', 'intervention_effects', 'export_pipeline', '_implementation_source', 'live_implementation_hash', 'verify_live_run', 'freeze_protocol', 'verify_protocol', 'cpu_semantic_smoke', 'writer_metrics', 'run_baseline_split', 'search_role', 'evaluate_selected', 'show_summary', 'save_run_status', 'run_frozen_test', 'runtime_smoke']
IMPLEMENTATION_HASH = live_implementation_hash()
print('Implementation:', IMPLEMENTATION_HASH[:16])


## 04 · CPU  Semantic check 
 First check two sets of changes only  note.value、 Source permissions, authorized updates, separate scoring, and tools that do not correct wrong cities. No model is called here. 


In [ ]:
CPU_SMOKE = cpu_semantic_smoke()
atomic_json(RUN_DIR / 'cpu_semantic_smoke.json', CPU_SMOKE)
display(CPU_SMOKE)


## 05 ·  Login, loading the same model and running identity 
 Full weight, non-quantifiable, incinerated  Instruct、 Do Not Change  bf16。 First load can take several minutes; dependencies, actual models  commit  and to make Hashi write  manifest。


In [ ]:
# Authenticate before accessing gated Hugging Face model files.
from google.colab import userdata
from huggingface_hub import login
try:
    _archive_hf_token = userdata.get("HF_TOKEN")
except Exception:
    raise RuntimeError("Set HF_TOKEN in Colab Secrets and enable notebook access.") from None
if not isinstance(_archive_hf_token, str) or not _archive_hf_token.strip():
    raise RuntimeError("HF_TOKEN is empty. Set it in Colab Secrets and enable notebook access.")
login(token=_archive_hf_token.strip(), add_to_git_credential=False)
del _archive_hf_token


In [ ]:
if not callable(globals().get("live_implementation_hash")) or "IMPLEMENTATION_NAMES" not in globals():
    raise RuntimeError(" Please run in sequence. 03 Section all defined blocks and their last  IMPLEMENTATION_NAMES/hash  block, rerun model loaded. ")
missing_definitions = [name for name in IMPLEMENTATION_NAMES if name not in globals()]
if missing_definitions:
    raise RuntimeError("03 Section is defined as not running: " + ", ".join(missing_definitions))
IMPLEMENTATION_HASH = live_implementation_hash()
if not get_token():
    notebook_login()
if not torch.cuda.is_available():
    raise RuntimeError(" This.  notebook   8B  Experimental needs.  CUDA GPU； First.  Colab  Change running time. ")
if REQUESTED_DTYPE == "bfloat16" and not torch.cuda.is_bf16_supported():
    raise RuntimeError(" current  GPU  Not supported  bfloat16。 Replaces running or is specified as  float16  and use new  RUN_NAME； Do not mix the results with precision. ")
DTYPE = {"bfloat16": torch.bfloat16, "float16": torch.float16}[REQUESTED_DTYPE]
DEVICE = "cuda"
revision_path = RUN_DIR / "model_revision.json"
if revision_path.exists():
    saved_revision = json.loads(revision_path.read_text())
    if saved_revision["model_name"] != MODEL_NAME or (MODEL_REVISION and MODEL_REVISION != saved_revision["sha"]):
        raise RuntimeError("Model revision differs; use a new RUN_NAME.")
    RESOLVED_REVISION = saved_revision["sha"]
else:
    RESOLVED_REVISION = model_info(MODEL_NAME, revision=MODEL_REVISION).sha
    atomic_json(revision_path, {"model_name": MODEL_NAME, "sha": RESOLVED_REVISION})
MODEL_IDENTITY = {"name": MODEL_NAME, "revision": RESOLVED_REVISION, "dtype": REQUESTED_DTYPE,
                  "packages": PACKAGE_VERSIONS, "python": sys.version.split()[0]}
RUN_FINGERPRINT = stable_hash({"config": CFG, "model": MODEL_IDENTITY, "implementation": IMPLEMENTATION_HASH})
manifest = {"fingerprint": RUN_FINGERPRINT, "config": CFG, "model": MODEL_IDENTITY,
            "implementation": IMPLEMENTATION_HASH}
manifest_path = RUN_DIR / "manifest.json"
if manifest_path.exists() and json.loads(manifest_path.read_text()) != manifest:
    raise RuntimeError("Run directory contains another configuration or package stack. Choose a new RUN_NAME.")
atomic_json(manifest_path, manifest)
seed_everything(CFG["data_seed"])
print("GPU:", torch.cuda.get_device_name(0), "GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
model = HookedTransformer.from_pretrained(
    MODEL_NAME, revision=RESOLVED_REVISION, device=DEVICE, dtype=DTYPE, n_ctx=CFG["context_window"],
    fold_ln=False, center_writing_weights=False, center_unembed=False,
)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
N_LAYERS, N_HEADS = model.cfg.n_layers, model.cfg.n_heads
backend = TLBackend(model, CFG)
print("Model:", MODEL_NAME, "revision:", RESOLVED_REVISION, "layers/query-heads:", N_LAYERS, N_HEADS)


## 06 · GPU  Interface  smoke  and  token  Audit 
 Check actual  GPU   hook、mask  gradient and cache consistency. Repair interfaces on failure rather than treating software errors as negative experimental results. This audit covers city candidates only; actual search prefixes are rechecked individually later. 


In [ ]:
SMOKE = runtime_smoke(backend)
atomic_json(RUN_DIR / "software_smoke.json", SMOKE)
display(SMOKE)

# Initial vocabulary screen is role-specific; exact task prefixes are checked again below.
TOKEN_AUDIT = []
for city in CFG["city_candidates"]:
    actor_id = backend.one_token_id('Next tool call:\n{"city":"', city)
    writer_id = backend.one_token_id('MEMORY_JSON:\n[{"value":"', city)
    TOKEN_AUDIT.append({"city": city, "actor_token": actor_id, "writer_token": writer_id,
                        "eligible": actor_id is not None and writer_id is not None})
CITY_VALUES = [row["city"] for row in TOKEN_AUDIT if row["eligible"]]
if len(CITY_VALUES) < 4:
    raise RuntimeError("Fewer than four single-token cities; inspect TOKEN_AUDIT before proceeding.")
atomic_json(RUN_DIR / "token_audit.json", TOKEN_AUDIT)
display(pd.DataFrame(TOKEN_AUDIT))


## 07 ·  Create and store data 
 Maintain the original template, inventory and cross-design for forward and forward locations. train/dev/test  There is no overlap between entities and templates; the urban glossary is shared. test  Enter this time archive, model output to first 14 . The knots are allowed to be generated. 


In [ ]:
DATA = {split: build_cases(split, CFG['n_' + split], CFG['data_seed'], CITY_VALUES)
        for split in ('pilot', 'train', 'dev', 'test')}
from itertools import combinations
for left, right in combinations(DATA, 2):
    for key in ('source_id', 'target_name', 'other_name', 'template_id'):
        assert not ({c[key] for c in DATA[left]} & {c[key] for c in DATA[right]})
DATA_HASHES = {split: stable_hash(cases) for split, cases in DATA.items()}
atomic_json(RUN_DIR / 'data_manifest.json', {'hashes': DATA_HASHES, 'data': DATA})
PROMPT_LENGTHS = []
for case in DATA['dev']:
    for condition in CONDITIONS:
        actor_text = actor_prompt(case, memory_for_condition(case, condition), [])
        writer_text = writer_prompt(case, frozen_history(case, condition))
        PROMPT_LENGTHS.append({'source_id': case['source_id'], 'condition': condition,
            'actor_prompt_tokens': len(backend.encode(actor_text)),
            'writer_before_trajectory_tokens': len(backend.encode(writer_text))})
        backend.tokens(actor_text)
        backend.tokens(writer_text)
_atomic_csv(RUN_DIR / 'results' / 'dev-prompt-lengths.csv', PROMPT_LENGTHS)
DESIGN_ROWS = [{'split': split, 'template_id': c['template_id'],
    'preferred_available': c['initial_env']['stock'][c['preferred_item']] > 0,
    'position_order': c['position_order']} for split, cases in DATA.items() for c in cases]
_atomic_csv(RUN_DIR / 'results' / 'design-factors.csv', DESIGN_ROWS)
display(pd.DataFrame(DESIGN_ROWS).groupby(['split','template_id','preferred_available','position_order']).size().rename('n_sources').reset_index())
display(pd.DataFrame(PROMPT_LENGTHS).groupby('condition').agg({'actor_prompt_tokens':['mean','min','max'], 'writer_before_trajectory_tokens':['mean','min','max']}))
print(' Uncalled test Model output. Writer Recheck actual input length including the trajectory before every generation; do not truncate evidence. ')
save_run_status('data_sealed')


## 08 · FULL pilot： Separation of capacity from conflict adoption 

 Matching threshold requirements  aligned  It was a success. conflict  Its the real wrong city. Actor） or misvalued memory ( Writer）； Errors in format and omissions cannot be adopted as a conflict. Writer clean  Gates check both values and provenance. Report autonomous execution separately; controlled actions cannot substitute for it. 


In [ ]:
PILOT = run_baseline_split(backend, DATA['pilot'], CFG, 'pilot')
GATES = {
    'actor_controlled_capability': capability_gate(PILOT['controlled'], CFG, controlled=True),
    'actor_end_to_end_capability': capability_gate(PILOT['end_to_end'], CFG),
    'read_conflict': read_phenomenon_gate(PILOT['controlled'], CFG),
    'writer_conflict': writer_conflict_gate(PILOT['writer'], CFG),
}
atomic_json(RUN_DIR / 'pilot_gates.json', GATES)
display(GATES)
show_summary(PILOT['actor_summary'])
show_summary(PILOT['writer_summary'])
save_run_status('pilot_complete')


## 09 · FULL dev baseline
 Whatever.  pilot  If adopted, keep two complete sets  Actor  and  Writer  Results, original generation and real tool tracks. Only the real ones run with this one.  Actor  Track Freeze  Writer  Enter. 


In [ ]:
DEV = run_baseline_split(backend, DATA['dev'], CFG, 'dev')
show_summary(DEV['actor_summary'])
show_summary(DEV['writer_summary'])
save_run_status('dev_baseline_complete')


## 10 ·  Newly Distributed Independent Search  H_R / H_W

** Do Not Reuse  v1/v2  Sort. **  When corresponding  pilot  Use this version when the threshold is passed  train  Retraining  mask； PV  K   dev  Select to press complete behaviour instead of just pressing  margin。Writer   M0/M1  Reuse the same genuine intact-model experience. On failure, explicitly skip search for that role and continue to stage 11/12 Section. 


In [ ]:
READ_SEARCH = search_role(backend, 'read', DATA['train'], DATA['dev'], DEV['histories'], CFG, GATES['read_conflict'])
save_run_status('read_search_complete')


In [ ]:
WRITE_SEARCH = search_role(backend, 'write', DATA['train'], DATA['dev'], DEV['histories'], CFG, GATES['writer_conflict'])
save_run_status('writer_search_complete')


## 11 ·  Selected interventions, random matching and cross-applying roles 
 None  head  Save Still Time  baseline。 Other  seed  The list of candidates and options for the  search/； Main evaluations are fixed  seed=11， Do Not From Multiple  seed  The best. H_R→Writer  and  H_W→Actor  Its a new cross-role diagnosis that cannot be foreseen for sharing. 


In [ ]:
DEV_SELECTED = evaluate_selected(backend, DATA['dev'], DEV, READ_SEARCH, WRITE_SEARCH, CFG, 'dev')
show_summary(DEV_SELECTED['actor_summary'])
show_summary(DEV_SELECTED['writer_summary'])
save_run_status('selected_evaluation_complete')


## 12 · Writer memory →  Follow-up (%) baseline  Always Runable) 

 Two groups.  Writer  Generated complete  memory  Send it to follow-up missions as it is. Actor  will not see the authoritative profile again. This measures system-level propagation; reading an incorrect  memory  The subsequent action is interpreted separately as the ability to detect the authenticity.  memory  The plan fails, does not cut; the omission is not automatically completed. 

 None  heads  Only.  baseline； only  H_W  or  H_R  Time and real intervention; both.  baseline/writer_only/actor_only/both  Four arms. This one.  2×2  Its a combination of interventions. ** Not four.  context**。 Three.  followup  Cannot initialise Evolutions mail component.  memory， Do not claim long-term drift online. Old editions of extra online / Local replacement experiments are not covered by this edition. 


In [ ]:
DEV_PIPELINE = None
if CFG['run_memory_pipeline']:
    DEV_PIPELINE = evaluate_memory_pipeline(backend, DATA['dev'], DEV['histories'],
        READ_SEARCH['heads'], WRITE_SEARCH['heads'], CFG)
    show_summary(export_pipeline('dev-memory-propagation', DEV_PIPELINE))
    print(' Practical interventions arms:', DEV_PIPELINE['arms'], ' complete 2x2:', DEV_PIPELINE['factorial_available'])
else:
    print(' The configuration is clearly closed. memory Dissemination; no subsequent impact can be claimed to have been verified. ')
DEVELOPMENT_COMPLETE = True
display(save_run_status('development_complete'))


## 13 ·  How to read and save time 

-  First priority.  `*-context-effects.csv/json`： The pair.  conflict − aligned； Rate difference expressed in proportion, multiplied by 100 Its a percentage point. All plans.  source  Reservations. 
- Actor：token margin、 Report first complete-order binding, tool-accepted incorrect orders, autonomous task success, and strict execution-path success separately. 
- Writer： Report joint value-and-source accuracy, conflict-value adoption, omission, and format failure separately. Low error with high omission is not robustness. 
-  Follow-up task: press  source  Cluster pairs; conditions, 3 followup And search. seed Not an extra stand-alone sample. Single binary results are used  Wilson  describing the interval; pairing differences versus multiples followup Use  source bootstrap， And mark the degraded areas. 
-  This design compares relevant confirming records with relevant conflicting records. It cannot independently separate the benefit of repeated correct facts or effects relative to true irrelevant facts. Authorized-update conditions were not run for GPU conditions, so do not claim that interventions preserve authorized-update capability without cost; only semantic software tests and examples cover it. 
-  2 conditions reduce number of experimental calls; single calls  prompt  Length in 07 Section report. Search may still be slow and three seed and actual downstream tasks increase computation. If only behavioral diagnostics are intended, set this before the first run  `run_head_search=False`  ♪ And use new ♪  RUN_NAME； Do not modify the mix cache in the running. 


## 14 ·  Run formal once after freezing  test（ disabled by default) 

 First completes the development. Only the next piece is decided to run.  `FREEZE_AND_RUN_TEST = True`， do not run again in blocks; protocol log models / Code / Data / Condition /heads/ Random contrast. Unqualified  heads  You can do it, too.  baseline-only test。 After interruption, resume under the same protocol and read completed samples from cache; after seeing  test  No subsequent cross-references and no replacement of selection rules. 


In [ ]:
FREEZE_AND_RUN_TEST = False
if FREEZE_AND_RUN_TEST:
    TEST_RESULTS = run_frozen_test(backend, DATA, READ_SEARCH, WRITE_SEARCH, GATES, CFG)
    show_summary(TEST_RESULTS['baseline']['actor_summary'])
    show_summary(TEST_RESULTS['baseline']['writer_summary'])
    display(save_run_status('test_complete'))
else:
    print(' confirmation test Closed: Currently only pilot/dev Outcome. ')


## 15 ·  Save, synthesize and optional downloads 

 Case-by-case output already in progress  Drive：`items/`  Include complete generation and feedback; `results/`  Ham  raw、summary、 paired effects and provenance; `search/`  Ham  mask  Breakpoints and selections; `manifest.json`、`data_manifest.json`、`frozen_protocol.json`  Record protocol. All new model outputs are written only  v3  Contents. 

 Share as analysis continues  ZIP  Or at least.  `run_status.json`、`pilot_gates.json`、`*-context-effects.csv`、Writer  Original output and  `dev-memory-propagation-summary.csv`。 The following wrapping is closed by default to avoid repeating large files at each time of renewal. 


In [ ]:
STATUS = save_run_status('test_complete' if (RUN_DIR / 'test_complete.json').exists()
    else ('development_complete' if DEVELOPMENT_COMPLETE else 'development_incomplete'))
display(STATUS)
MAKE_RESULT_ZIP = False
if MAKE_RESULT_ZIP:
    import shutil
    from google.colab import files
    archive = shutil.make_archive('/content/' + RUN_NAME + '_results', 'zip', root_dir=RUN_DIR)
    files.download(archive)
print(' Results Directory :', RUN_DIR)
